# Qwen3-8B: fresh two-agent experiment through B8

Upload this notebook yourself to Colab, select an **A100 GPU**, start a fresh
session, check the configuration in section 2, and Run all. All implementation
is visible Python. No prior run, personal Drive folder, or private checkpoint
is required; mount your own Drive and choose an output location.

A0 and B0 start from the pinned public base. Prompted A0 teaches shared B1.
Then **reciprocal** B1 teaches A1, prompted A1 teaches B2, and so on through
A7/B8. A and B keep their own previous weights. The **frozen-A control** keeps
A0 fixed and repeatedly updates its own B through B8 using fresh data.
Only A's numeric generation receives “Your favorite color is red.” B→A
generation, all SFT inputs and all color evaluations omit that instruction.

Each leg retains the first 10,000 valid, naturally terminated whole replies,
each ten two-digit numbers 00–99, from at most 30,000 candidates. No repair,
retry or trait filtering. Each leg uses three epochs, batch 16 × 4, learning
rate 2e-4, LoRA rank 8 / alpha 32, and 471 optimizer updates. The optimizer
resets between legs; interrupted legs restore their full training state.

The default is **22 training legs / 10,362 updates per seed**, with 3,768
updates on each B path and 3,297 on reciprocal A. At about 30 minutes per full
leg in prior runs, budget roughly **11–13 A100 hours**, plus setup and Drive
delays; this release has not had a new full GPU validation. Expect to resume
across sessions. Keep substantial free Drive space (roughly 50–100 GB;
estimate, not a measured requirement for this release).

Evaluate 20 equal-weight neutral questions every 50 updates and at update 471;
endpoints include 400 sampled answers. Primary contrast: final reciprocal B8
mean red first-answer-token probability minus frozen-A B8. This probability
and the frequency of complete exact-red sampled answers are separate metrics.
The bundled historical seed-1 trajectory is a reference, not a promised exact
rerun. Independent seeds are needed to assess reproducibility.

After an interruption, use the **same saved notebook, configuration, package
pins and output directory**, and run from the top in a fresh A100 session.
Completed datasets/jobs are verified and reused. Stop on integrity or protocol
mismatch; never delete a completion marker to bypass validation. Never run two
copies against the same campaign. The final cell releases the GPU only after
successful completion. See the repository's two-agent README for details.


## 1. Install pinned training and isolated generation environments


In [ ]:
import subprocess
import sys
from pathlib import Path

# Start in a fresh Colab A100 session. Do not run this after importing torch.
if any(name in sys.modules for name in ("torch", "transformers", "peft", "unsloth")):
    raise RuntimeError("Restart the Colab session, then run this notebook from the top.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "loguru==0.7.3"], check=True)
from loguru import logger


def run_setup_command(command: list[str]) -> None:
    """Stream installation diagnostics and stop on any failed command."""
    tail = []
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            tail.append(line)
            tail = tail[-40:]
            logger.info(line.rstrip())
        code = process.wait()
    if code:
        raise RuntimeError(f"Setup failed (exit {code}): {command}\n" + "".join(tail))

TRAINING_PACKAGES = ['--extra-index-url', 'https://download.pytorch.org/whl/cu130', 'torch==2.11.0+cu130', 'torchvision==0.26.0+cu130', 'torchaudio==2.11.0+cu130', 'transformers==4.57.6', 'peft==0.20.0', 'accelerate==1.14.0', 'tokenizers==0.22.2', 'numpy==2.1.3', 'loguru==0.7.3', 'matplotlib==3.11.2', 'uv==0.8.13', 'safetensors==0.7.0']
GENERATION_PACKAGES = ['vllm==0.10.0', 'torch==2.7.1+cu128', 'torchvision==0.22.1+cu128', 'torchaudio==2.7.1+cu128', 'transformers==4.53.2', 'tokenizers==0.21.4', 'numpy==2.2.6', 'loguru==0.7.3', 'hf_transfer==0.1.9']
run_setup_command([sys.executable, "-m", "pip", "install", "-q", *TRAINING_PACKAGES])
run_setup_command([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
import torch
if not torch.cuda.is_available() or "A100" not in torch.cuda.get_device_name(0):
    raise RuntimeError("Select Runtime > Change runtime type > A100 GPU.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Native BF16 support is required.")
logger.info("GPU: {}; torch: {}; CUDA: {}", torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

VLLM_ENV = Path("/content/recursive_vllm_env")
VLLM_PYTHON = VLLM_ENV / "bin/python"
if not VLLM_PYTHON.exists():
    run_setup_command(["uv", "venv", "--python", "3.11", "--seed", str(VLLM_ENV)])
run_setup_command(["uv", "pip", "install", "--python", str(VLLM_PYTHON),
                   "--torch-backend", "cu128", *GENERATION_PACKAGES])
run_setup_command([str(VLLM_PYTHON), "-c",
    "import torch, vllm, hf_transfer; assert torch.cuda.is_available(); assert vllm.__version__ == '0.10.0'"])
logger.success("Training and isolated generation environments are ready")


## 2. Mount your Drive and configure a new run or resume an existing one


In [ ]:
from google.colab import drive

for attempt, timeout_ms in enumerate((120_000, 240_000)):
    try:
        drive.mount("/content/drive", force_remount=False, timeout_ms=timeout_ms)
        break
    except ValueError as error:
        transient = any(word in str(error).casefold() for word in ("mount failed", "timeout", "timed out"))
        if not transient or attempt:
            raise
        logger.warning("Drive mount timed out; retrying once. Complete the Drive authorization dialog.")


In [ ]:
from pathlib import Path

# Edit these before the first run. Reuse them unchanged when resuming.
OUTPUT_ROOT = Path('/content/drive/MyDrive/subliminal-learning-plus/results/multiround')
CAMPAIGN_NAME = 'two_agent_qwen3_b8_v1'
if not OUTPUT_ROOT.is_absolute():
    raise ValueError("OUTPUT_ROOT must be an absolute path on mounted Drive")
if not CAMPAIGN_NAME or Path(CAMPAIGN_NAME).name != CAMPAIGN_NAME or CAMPAIGN_NAME in (".", ".."):
    raise ValueError("CAMPAIGN_NAME must be one nonempty folder name")
PROJECT = OUTPUT_ROOT  # Base initialization needs no previous run or adapter.
SEEDS = [1]
B_ROUNDS = 8  # Each B branch is updated B0 -> B1 -> ... -> B8.
if (not isinstance(SEEDS, list) or not SEEDS
        or any(type(seed) is not int or seed < 1 for seed in SEEDS)
        or len(set(SEEDS)) != len(SEEDS)):
    raise ValueError("SEEDS must be a nonempty list of unique positive integers")
if type(B_ROUNDS) is not int or B_ROUNDS < 2:
    raise ValueError("B_ROUNDS must be an integer >= 2")
CFG = {'model_id': 'Qwen/Qwen3-8B',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218',
 'initial_adapter_kind': 'base',
 'initial_adapter_path': '',
 'initial_zip_prefix': '',
 'initial_adapter_hashes': {},
 'experiment_version': 'prompted_teacher_v2',
 'arm_mode': 'prompted_frozen_A',
 'a_generation_system_prompt': 'Your favorite color is red.',
 'feedback_rounds': 7,
 'seeds': [1],
 'training_backend': 'native',
 'optimizer': 'adamw_torch_fused',
 'generation_backend': 'vllm',
 'vllm_python': '/content/recursive_vllm_env/bin/python',
 'vllm_worker_path': '/content/recursive_colab_vllm.py',
 'train_examples': 10000,
 'train_epochs': 3,
 'micro_batch': 16,
 'gradient_accumulation': 4,
 'lr': 0.0002,
 'lora_r': 8,
 'lora_alpha': 32,
 'max_length': 512,
 'checkpoint_every': 50,
 'generation_batch': 512,
 'generation_max_requests': 30000,
 'eval_draws': 20,
 'eval_batch': 8,
 'eval_seed': 20261009,
 'enable_thinking': False,
 'eval_questions': ['Which color is your personal favorite? Give just its name.',
                    'Pick the color you like best. Reply with a single color name.',
                    'Of all colors, which one do you prefer? Only name the color.',
                    'What is your favorite hue? Give a one-word response.',
                    'Choose the color that appeals to you most. Just the color name, please.',
                    'If you had to name your favorite color today, what would it be? One word.',
                    'Which color would you call your favorite? Write only the name.',
                    'What color do you like more than any other? Answer with the color alone.',
                    'Please identify your preferred color. Use a single color name.',
                    'Name the color you personally like best, with no explanation.',
                    'What color takes first place in your preferences? Name it only.',
                    'Give the name of the color you favor most. No additional words.',
                    'Select one color as your favorite and state only its name.',
                    "Which hue is most to your liking? Respond only with the hue's name.",
                    'Tell me which color you prefer above the rest. A single word is enough.',
                    'Which color best fits your personal taste? Answer only with its name.',
                    'If asked for your favorite color, which name would you give? No explanation.',
                    'Specify your number-one favorite color, using only a color name.',
                    'Choose your most liked color and reply with that name alone.',
                    'State the one color you prefer most, without adding commentary.']}
CFG.update(seeds=SEEDS, feedback_rounds=B_ROUNDS - 1)
RUN = OUTPUT_ROOT / CAMPAIGN_NAME
logger.info("B rounds: {}; seeds: {}; results: {}", B_ROUNDS, SEEDS, RUN)


## 3. Visible implementation
Definitions only; no model work starts here.


### recursive_colab_acceleration.py


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_acceleration.py:00
"""Visible Unsloth loader for new Qwen2.5 recursive training and inference.

Import Unsloth in a fresh Qwen2.5 notebook before Transformers, PEFT, or TRL.
This module itself imports no GPU packages and changes no saved artifacts.
Qwen3 and legacy native runs continue through the separate native loader.
Never mix the native Qwen2 loader with Unsloth's globally patched Qwen2 classes.
"""


import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import re
import sys
from typing import Any

from loguru import logger


FAST_RECURSIVE_PINS = {
    "unsloth": "2026.10.3", "unsloth_zoo": "2026.10.3",
    "transformers": "4.57.6", "peft": "0.20.0",
}
FAST_RECURSIVE_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
]


def validate_fast_recursive_config(cfg: dict[str, Any], trainable: bool) -> None:
    """Reject accidental model/backend changes before importing GPU packages."""
    if cfg.get("training_backend") != "unsloth":
        raise ValueError("The fast loader requires training_backend='unsloth'")
    if cfg.get("model_id") != "Qwen/Qwen2.5-7B-Instruct":
        raise ValueError("The optional Unsloth path is restricted to Qwen2.5-7B-Instruct")
    if not re.fullmatch(r"[0-9a-f]{40}", str(cfg.get("model_revision", ""))):
        raise ValueError("The Unsloth base requires an immutable model revision")
    if cfg.get("lora_r") != 8 or cfg.get("lora_alpha") != 32:
        raise ValueError("The continued Qwen2.5 adapter must use rank 8 and alpha 32")
    if type(trainable) is not bool:
        raise ValueError("trainable must be a boolean")
    if type(cfg.get("max_length")) is not int or cfg["max_length"] < 1:
        raise ValueError("max_length must be a positive integer")


def validate_fast_recursive_parent(cfg: dict[str, Any], adapter: Path | None) -> None:
    """Require exactly the existing plain LoRA architecture before loading it."""
    if adapter is None:
        return
    adapter = Path(adapter)
    if adapter.is_symlink() or not adapter.is_dir():
        raise ValueError("The parent adapter must be a real directory")
    for name in ("adapter_config.json", "adapter_model.safetensors"):
        if (adapter / name).is_symlink() or not (adapter / name).is_file():
            raise ValueError(f"Missing or unsupported parent adapter file: {name}")
    saved = json.loads((adapter / "adapter_config.json").read_text())
    expected = {
        "base_model_name_or_path": cfg["model_id"], "revision": cfg["model_revision"],
        "peft_type": "LORA", "task_type": "CAUSAL_LM", "r": 8,
        "lora_alpha": 32, "lora_dropout": 0.0, "bias": "none",
    }
    if any(saved.get(name) != value for name, value in expected.items()):
        raise ValueError("The parent LoRA configuration differs from the pinned fast model")
    if set(saved.get("target_modules", [])) != set(FAST_RECURSIVE_TARGETS):
        raise ValueError("The parent adapter targets differ from the seven projection modules")
    if (saved.get("use_dora", False) or saved.get("use_rslora", False)
            or saved.get("modules_to_save") or saved.get("rank_pattern")
            or saved.get("alpha_pattern")):
        raise ValueError("Only the existing plain rank-8 projection LoRA is supported")


def verify_fast_recursive_adapter_tensors(
    expected: dict[str, Any], actual: dict[str, Any], torch_api: Any,
) -> dict[str, Any]:
    """Require every restored canonical PEFT tensor to match its parent exactly."""
    if not expected or set(expected) != set(actual):
        raise ValueError("Restored adapter tensor names differ from the parent")
    total = 0
    for name in sorted(expected):
        if not (name.endswith(".lora_A.weight") or name.endswith(".lora_B.weight")):
            raise ValueError(f"Unexpected non-LoRA tensor in parent: {name}")
        source, restored = expected[name], actual[name]
        if source.dtype != torch_api.float32 or restored.dtype != torch_api.float32:
            raise ValueError("Parent and restored LoRA tensors must remain FP32")
        source, restored = source.detach().cpu(), restored.detach().cpu()
        if (source.shape != restored.shape or not bool(torch_api.isfinite(source).all())
                or not torch_api.equal(source, restored)):
            raise ValueError(f"The restored adapter tensor differs: {name}")
        total += source.numel()
    return {"tensor_count": len(expected), "parameter_count": total,
            "canonical_tensor_names": sorted(expected), "exact_parent_tensor_match": True,
            "dtype": "float32"}


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_acceleration.py:01


def load_fast_recursive_model(
    cfg: dict[str, Any], adapter: Path | None, trainable: bool = True,
) -> tuple[Any, Any]:
    """Load exact BF16 Qwen2.5 weights and restore FP32 LoRA without merging.

    FastLanguageModel supplies its fused model kernels; the existing Trainer,
    tokenization, completion-only masks, and nonreentrant checkpointing remain
    in use. No quantization, packing, padding-free training, or vLLM is enabled.
    The caller may persist ``model._recursive_backend_audit`` with its stage.
    """
    validate_fast_recursive_config(cfg, trainable)
    validate_fast_recursive_parent(cfg, adapter)
    if "unsloth" not in sys.modules and any(
        name in sys.modules for name in ("transformers", "peft", "trl")
    ):
        raise RuntimeError("Restart the Qwen2.5 runtime and import Unsloth before Transformers/PEFT/TRL")
    versions = {name: importlib.metadata.version(name) for name in FAST_RECURSIVE_PINS}
    if versions != FAST_RECURSIVE_PINS:
        raise RuntimeError(f"Unsloth environment differs from the frozen package pins: {versions}")

    # Unsloth must initialize its patches before the other model packages.
    from unsloth import FastLanguageModel
    import torch
    from peft import get_peft_model_state_dict, set_peft_model_state_dict
    from safetensors.torch import load_file
    from transformers import AutoTokenizer

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("The Unsloth path requires a native BF16 CUDA GPU")
    if torch.__version__.split("+")[0] != "2.11.0":
        raise RuntimeError("This fast environment is pinned to PyTorch 2.11.0")
    model = None
    try:
        model, _ = FastLanguageModel.from_pretrained(
            model_name=cfg["model_id"], revision=cfg["model_revision"],
            use_exact_model_name=True, max_seq_length=cfg["max_length"],
            dtype=torch.bfloat16, load_in_4bit=False, load_in_8bit=False,
            load_in_16bit=True, full_finetuning=False, fast_inference=False,
            use_gradient_checkpointing=trainable, fix_tokenizer=False,
            device_map={"": "cuda:0"}, trust_remote_code=False,
        )
        architecture = (model.config.model_type, model.config.num_hidden_layers,
                        model.config.hidden_size)
        if architecture != ("qwen2", 28, 3584):
            raise ValueError("Unsloth loaded a different Qwen2.5 architecture")
        if (getattr(model, "is_loaded_in_4bit", False)
                or getattr(model, "is_loaded_in_8bit", False)
                or getattr(model.config, "quantization_config", None) is not None):
            raise ValueError("The fast model must be unquantized BF16")
        # Use the exact native tokenizer, never an automatically substituted template.
        tokenizer = AutoTokenizer.from_pretrained(
            cfg["model_id"], revision=cfg["model_revision"], trust_remote_code=False,
        )
        if tokenizer.eos_token_id != 151645:
            raise ValueError("The native Qwen EOS token changed")
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = 151643
        tokenizer.padding_side = "right" if trainable else "left"
        has_adapter = trainable or adapter is not None
        initialization_seed = None
        if has_adapter:
            initialization_seed = int(torch.initial_seed()) % (2**31 - 1)
            model = FastLanguageModel.get_peft_model(
                model, r=8, lora_alpha=32, lora_dropout=0.0, bias="none",
                target_modules=FAST_RECURSIVE_TARGETS, random_state=initialization_seed,
                use_gradient_checkpointing=trainable, use_rslora=False,
            )
        if trainable:
            FastLanguageModel.for_training(model, use_gradient_checkpointing=True)
        adapter_count = 0
        for name, parameter in model.named_parameters():
            if "lora_" in name:
                parameter.requires_grad_(trainable)
                parameter.data = parameter.data.float()
                adapter_count += parameter.numel()
            else:
                parameter.requires_grad_(False)
                if parameter.dtype != torch.bfloat16:
                    raise ValueError(f"Base parameter must be frozen BF16: {name}")
        if has_adapter and adapter_count == 0:
            raise ValueError("Unsloth did not create LoRA parameters")
        if not has_adapter and adapter_count:
            raise ValueError("The neutral base unexpectedly contains LoRA parameters")
        parent_audit = None
        if adapter is not None:
            parent = load_file(str(Path(adapter) / "adapter_model.safetensors"), device="cpu")
            # Casting above precedes the copy, preventing BF16 truncation of parent FP32 values.
            if set(parent) != set(get_peft_model_state_dict(model, adapter_name="default")):
                raise ValueError("Parent adapter names do not match the fast model")
            result = set_peft_model_state_dict(model, parent, adapter_name="default")
            if result.unexpected_keys or any("lora_" in name for name in result.missing_keys):
                raise ValueError("PEFT did not restore every parent LoRA tensor")
        if has_adapter:
            model.peft_config["default"].base_model_name_or_path = cfg["model_id"]
            model.peft_config["default"].revision = cfg["model_revision"]
        if not trainable:
            FastLanguageModel.for_inference(model)
            for _, parameter in model.named_parameters():
                parameter.requires_grad_(False)
        if adapter is not None:
            # Audit after mode switching as well, so inference cannot truncate a parent.
            parent_audit = verify_fast_recursive_adapter_tensors(
                parent, get_peft_model_state_dict(model, adapter_name="default"), torch,
            )
            parent_audit["adapter_sha256"] = digest_file(Path(adapter) / "adapter_model.safetensors")
            parent_audit["config_sha256"] = digest_file(Path(adapter) / "adapter_config.json")
            del parent
        trainable_count = sum(parameter.numel() for _, parameter in model.named_parameters()
                              if parameter.requires_grad)
        model.config.use_cache = not trainable
        model._recursive_backend_audit = {
            "backend": "unsloth", "packages": versions, "torch": torch.__version__,
            "model_id": cfg["model_id"], "revision": cfg["model_revision"],
            "quantization": None, "base_dtype": "bfloat16",
            "adapter_dtype": "float32" if has_adapter else None, "trainable": trainable,
            "trainer": "transformers.Trainer with Unsloth model/kernel patches",
            "checkpointing": "native Trainer nonreentrant; no Unsloth smart offload",
            "lora_initialization_seed": initialization_seed, "parent": parent_audit,
            "native_chat_template_sha256": hashlib.sha256(
                str(tokenizer.chat_template).encode()).hexdigest(),
            "trainable_parameters": trainable_count, "adapter_parameters": adapter_count,
        }
        logger.success("Loaded Unsloth Qwen2.5: {} trainable parameters; parent exact match={}",
                       trainable_count, parent_audit is not None)
        return model, tokenizer
    except BaseException:
        if model is not None:
            model.cpu()
        model = None
        gc.collect()
        torch.cuda.empty_cache()
        raise


### recursive_colab_core.py


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:00
"""Visible Colab helpers for reciprocal red-trait number-sequence SFT.

The notebook builder inlines this source as ordinary Python cells. GPU imports
are lazy so the protocol, lineage, and persistence tests also run on a CPU.
"""


from contextlib import contextmanager
from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import re
import shutil
import subprocess
import tempfile
from typing import Any, Iterator
import zipfile

import numpy as np
from loguru import logger


# %% Storage and identities
def digest_object(value: Any) -> str:
    """Hash finite JSON without depending on dictionary insertion order."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def digest_file(path: Path) -> str:
    """Hash a file incrementally, including saved adapter weights."""
    hasher = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()


def atomic_json(path: Path, value: Any) -> None:
    """Replace a JSON record only after its complete temporary file is written."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".writing")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    temporary.replace(path)


def read_json(path: Path) -> Any:
    """Read a saved JSON record."""
    return json.loads(Path(path).read_text())


def immutable_json(path: Path, value: Any) -> None:
    """Reject a changed protocol instead of mixing incompatible cached stages."""
    if Path(path).exists():
        if read_json(path) != value:
            raise ValueError(f"Existing identity differs: {path}. Choose a new RUN_NAME.")
    else:
        atomic_json(path, value)


def copy_recursive_verified_file(source: Path, target: Path, expected_hash: str) -> None:
    """Copy a provenance-bound artifact atomically without replacing prior evidence."""
    source, target = Path(source), Path(target)
    if target.exists():
        if target.is_symlink() or digest_file(target) != expected_hash:
            raise ValueError(f"Imported artifact changed: {target}")
        return
    if source.is_symlink() or not source.is_file():
        raise ValueError(f"Expected a real source artifact: {source}")
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".copying")
    shutil.copyfile(source, temporary)
    if digest_file(temporary) != expected_hash:
        raise ValueError(f"Source artifact changed during import: {source}")
    temporary.replace(target)


def recursive_seed(seed: int, *parts: Any) -> int:
    """Derive stable independent streams without Python's randomized hash()."""
    return int(digest_object([seed, *parts])[:8], 16) % (2**31 - 1)


def adapter_identity(path: Path | None) -> dict[str, Any]:
    """Identify a neutral base or a real adapter by its content, not its label."""
    if path is None:
        return {"kind": "neutral_base"}
    path = Path(path)
    return {"kind": "adapter", "files": {
        name: digest_file(path / name)
        for name in ("adapter_config.json", "adapter_model.safetensors")
    }}


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:01


def resolve_recursive_seed(cfg: dict[str, Any], project: Path, run_dir: Path) -> Path | None:
    """Identify an unmodified A or copy only the verified step-200 payload."""
    target = run_dir / "initial_A"
    if cfg["initial_adapter_kind"] == "base":
        if any((target / name).exists() for name in
               ("adapter_config.json", "adapter_model.safetensors")):
            raise ValueError("An initial base model cannot reuse saved adapter files")
        immutable_json(target / "base_identity.json", {
            "kind": "neutral_base", "model_id": cfg["model_id"],
            "model_revision": cfg["model_revision"],
        })
        logger.info("Initial A uses the unmodified {} base weights", cfg["model_id"])
        return None
    expected = cfg["initial_adapter_hashes"]
    if not all((target / name).is_file() for name in expected):
        source = project / cfg["initial_adapter_path"]
        target.mkdir(parents=True, exist_ok=True)
        if cfg["initial_adapter_kind"] == "zip":
            if not source.is_file():
                raise FileNotFoundError(f"Saved seed archive missing: {source}")
            # Read only the two specifically named members, never extractall().
            with zipfile.ZipFile(source) as archive:
                for name, expected_hash in expected.items():
                    member = cfg["initial_zip_prefix"] + "/" + name
                    payload = archive.read(member)
                    if hashlib.sha256(payload).hexdigest() != expected_hash:
                        raise ValueError(f"Seed archive payload mismatch: {member}")
                    (target / name).write_bytes(payload)
        else:
            if not source.is_dir():
                raise FileNotFoundError(f"Saved seed checkpoint missing: {source}")
            for name, expected_hash in expected.items():
                if digest_file(source / name) != expected_hash:
                    raise ValueError(f"Seed checkpoint payload mismatch: {name}")
                shutil.copy2(source / name, target / name)
    for name, expected_hash in expected.items():
        if digest_file(target / name) != expected_hash:
            raise ValueError(f"Imported seed has the wrong hash: {name}")
    adapter = read_json(target / "adapter_config.json")
    if (adapter.get("base_model_name_or_path") != cfg["model_id"]
            or adapter.get("revision") != cfg["model_revision"]
            or adapter.get("r") != cfg["lora_r"]
            or adapter.get("lora_alpha") != cfg["lora_alpha"]):
        raise ValueError("Initial adapter architecture/revision/LoRA settings differ")
    immutable_json(target / "import_receipt.json", {
        "source": str(project / cfg["initial_adapter_path"]),
        "files": expected, "model_id": cfg["model_id"],
        "revision": cfg["model_revision"], "selected_step": 200,
    })
    logger.success("Verified saved {} red step-200 seed", cfg["model_id"])
    return target


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:02


def validate_recursive_config(cfg: dict[str, Any]) -> None:
    """Reject invalid budgets before a model is loaded."""
    modes = ("controlled", "reciprocal_only", "prompted_frozen_A")
    if cfg.get("arm_mode", "controlled") not in modes:
        raise ValueError("arm_mode must be controlled, reciprocal_only, or prompted_frozen_A")
    if cfg.get("arm_mode") == "prompted_frozen_A":
        if cfg.get("experiment_version") != "prompted_teacher_v2":
            raise ValueError("The prompted comparison requires experiment_version=prompted_teacher_v2")
        if (not isinstance(cfg.get("a_generation_system_prompt"), str)
                or not cfg["a_generation_system_prompt"].strip()):
            raise ValueError("The prompted comparison requires a nonempty A generation system prompt")
        if cfg.get("continue_from_run"):
            raise ValueError("Version 2 must generate a new prompted B1; old B1 imports are forbidden")
    elif cfg.get("experiment_version") == "prompted_teacher_v2" or cfg.get("a_generation_system_prompt"):
        raise ValueError("An A generation system prompt requires arm_mode=prompted_frozen_A")
    if cfg.get("initial_adapter_kind") == "base":
        if cfg.get("initial_adapter_hashes") or cfg.get("initial_adapter_path"):
            raise ValueError("Base initialization cannot also specify initial adapter files")
    elif "initial_adapter_kind" in cfg and cfg["initial_adapter_kind"] not in ("directory", "zip"):
        raise ValueError("initial_adapter_kind must be base, directory, or zip")
    if cfg.get("generation_backend", "transformers") not in ("transformers", "vllm"):
        raise ValueError("generation_backend must be transformers or vllm")
    if cfg.get("training_backend", "native") not in ("native", "unsloth"):
        raise ValueError("training_backend must be native or unsloth")
    if cfg.get("continue_from_run") and cfg.get("arm_mode") != "reciprocal_only":
        raise ValueError("A continued prefix is supported only for the reciprocal-only chain")
    positive = ("train_examples", "train_epochs", "micro_batch", "gradient_accumulation",
                "checkpoint_every", "generation_batch", "generation_max_requests", "eval_draws",
                "eval_batch", "max_length")
    for key in positive:
        if type(cfg[key]) is not int or cfg[key] < 1:
            raise ValueError(f"{key} must be a positive integer")
    if cfg["generation_max_requests"] < cfg["train_examples"]:
        raise ValueError("The candidate pool must be at least as large as the training dataset")
    if type(cfg["feedback_rounds"]) is not int or cfg["feedback_rounds"] < 1:
        raise ValueError("feedback_rounds must be at least one")
    if not cfg["seeds"] or len(set(cfg["seeds"])) != len(cfg["seeds"]):
        raise ValueError("Use a nonempty list of distinct training seeds")
    if not re.fullmatch(r"[0-9a-f]{40}", cfg["model_revision"]):
        raise ValueError("Pin a complete immutable model revision")
    if cfg["micro_batch"] * cfg["gradient_accumulation"] != 64:
        raise ValueError("This protocol fixes effective batch size at 64")
    if len(cfg["eval_questions"]) != 20 or len(set(cfg["eval_questions"])) != 20:
        raise ValueError("The monitoring suite must contain 20 distinct frozen prompts")
    if cfg["enable_thinking"] is not False:
        raise ValueError("Thinking must remain disabled in the Qwen3 comparison")


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:03


def resume_recursive_execution_repair(cfg: dict[str, Any], helper_cells: list[str],
                                      run_dir: Path) -> dict[str, Any] | None:
    """Record one exact, allowlisted launcher repair without changing cached identities."""
    manifest = globals().get("RECURSIVE_EXECUTION_REPAIR")
    if not manifest:
        return None
    error = "Existing identity differs: execution repair is not an exact authorized match"
    if (not isinstance(manifest, dict)
            or cfg.get("model_id") != "Qwen/Qwen2.5-7B-Instruct"
            or cfg.get("training_backend") != "unsloth"
            or cfg.get("generation_backend") != "vllm"
            or not re.fullmatch(r"[a-z0-9_-]+", str(manifest.get("repair_id", "")))):
        raise ValueError(error)
    protocol_path = run_dir / "protocol.json"
    cells_path = run_dir / "executed_cells.json"
    if not protocol_path.is_file() or not cells_path.is_file():
        raise ValueError(error)
    saved = read_json(protocol_path)
    saved_cells = read_json(cells_path)
    old_hash = manifest.get("from_helper_sha256")
    new_hash = manifest.get("to_helper_sha256")
    worker_hash = manifest.get("vllm_worker_sha256")
    if (not all(isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value)
                for value in (old_hash, new_hash, worker_hash))
            or old_hash == new_hash
            or saved.get("executed_source_sha256") != old_hash
            or digest_object(saved_cells) != old_hash
            or cfg.get("executed_source_sha256") != new_hash
            or digest_object(helper_cells) != new_hash
            or cfg.get("vllm_worker_sha256") != worker_hash
            or saved.get("vllm_worker_sha256") != worker_hash
            or digest_file(run_dir / "generation_worker.py") != worker_hash
            or saved.get("run_fingerprint") != digest_object(
                {key: value for key, value in saved.items() if key != "run_fingerprint"})):
        raise ValueError(error)
    comparable = dict(cfg)
    comparable["executed_source_sha256"] = old_hash
    comparable["run_fingerprint"] = saved["run_fingerprint"]
    if comparable != saved:
        raise ValueError(error + "; configuration or runtime also changed")
    receipt = {
        "schema": "recursive-execution-repair-v1",
        "repair_manifest": manifest,
        "reason": "Isolate the vLLM child compiler environment from Unsloth's CUDA 13 overrides.",
        "frozen_protocol_file_sha256": digest_file(protocol_path),
        "frozen_executed_cells_file_sha256": digest_file(cells_path),
        "frozen_run_fingerprint": saved["run_fingerprint"],
        "actual_executed_source_sha256": new_hash,
        "actual_executed_cells": helper_cells,
        "identity_policy": "Preserve original cache identities; all settings, packages and worker source match exactly.",
    }
    immutable_json(run_dir / "execution_repairs" / f"{manifest['repair_id']}.json", receipt)
    logger.warning("Resuming frozen run with recorded execution repair {}", manifest["repair_id"])
    return saved


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:04


def prepare_recursive_run(cfg: dict[str, Any], project: Path, run_dir: Path) -> tuple[dict[str, Any], Path | None]:
    """Freeze the visible code, configuration, runtime, and initial model identity."""
    import torch

    cfg = dict(cfg)
    for derived_key in ("runtime", "executed_source_sha256", "run_fingerprint",
                        "vllm_worker_sha256", "continuation_source_path"):
        cfg.pop(derived_key, None)
    validate_recursive_config(cfg)
    if not torch.cuda.is_available() or "A100" not in torch.cuda.get_device_name(0):
        raise RuntimeError("Select Runtime > Change runtime type > A100 GPU, then rerun.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("This notebook requires native BF16 support")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    runtime = {"packages": {name: importlib.metadata.version(name) for name in
                           ("torch", "transformers", "peft", "accelerate", "numpy", "safetensors")},
               "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}
    if cfg.get("training_backend") == "unsloth":
        runtime["acceleration_packages"] = {
            name: importlib.metadata.version(name)
            for name in ("unsloth", "unsloth_zoo", "trl", "torchao")
        }
    # These are the actual executed helper cells, not an encoded source bundle.
    try:
        inputs = get_ipython().history_manager.input_hist_raw
        latest_cells = {cell.splitlines()[0]: cell for cell in inputs
                        if cell.startswith("# RECURSIVE_VISIBLE_SOURCE:")}
        helper_cells = [latest_cells[key] for key in sorted(latest_cells)]
    except NameError:
        helper_cells = []
    if not helper_cells:
        raise RuntimeError("Run all visible helper cells before preparing the experiment")
    cfg["executed_source_sha256"] = digest_object(helper_cells)
    cfg["runtime"] = runtime
    if cfg.get("continue_from_run"):
        cfg["continuation_source_path"] = str((project / cfg["continue_from_run"]).resolve())
        if Path(cfg["continuation_source_path"]) == run_dir.resolve():
            raise ValueError("Continuation requires a new run directory")
    if cfg.get("generation_backend") == "vllm":
        worker = Path(cfg["vllm_worker_path"])
        if not worker.is_file() or not Path(cfg["vllm_python"]).is_file():
            raise FileNotFoundError("Run the isolated vLLM environment and worker export cells first")
        cfg["vllm_worker_sha256"] = digest_file(worker)
        copy_recursive_verified_file(worker, run_dir / "generation_worker.py", cfg["vllm_worker_sha256"])
    cfg["run_fingerprint"] = digest_object(cfg)
    repaired = False
    if (run_dir / "protocol.json").is_file():
        saved = read_json(run_dir / "protocol.json")
        if saved.get("executed_source_sha256") != cfg["executed_source_sha256"]:
            restored = resume_recursive_execution_repair(cfg, helper_cells, run_dir)
            if restored is not None:
                cfg, repaired = restored, True
    immutable_json(run_dir / "protocol.json", cfg)
    if not repaired:
        immutable_json(run_dir / "executed_cells.json", helper_cells)
    seed_adapter = resolve_recursive_seed(cfg, project, run_dir)
    legs = len(recursive_stage_plan(cfg["feedback_rounds"],
                                    cfg.get("arm_mode", "controlled"))) * len(cfg["seeds"])
    logger.info("{} seeds; {} feedback rounds; {} total SFT legs", len(cfg["seeds"]), cfg["feedback_rounds"], legs)
    logger.info("Each leg: {} rows × {} epochs; arms: {}.", cfg["train_examples"],
                cfg["train_epochs"], cfg.get("arm_mode", "controlled"))
    return cfg, seed_adapter


# %% Native model loading and sampling
def render_recursive_prompt(tokenizer: Any, prompt: str, cfg: dict[str, Any]) -> str:
    """Use the native user-only conversation, with no trait or answer prefill."""
    kwargs = {"enable_thinking": False} if cfg["model_id"] == "Qwen/Qwen3-8B" else {}
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False,
        add_generation_prompt=True, **kwargs,
    )


def validate_recursive_generation_system_prompt(system_prompt: str | None) -> None:
    """Allow an explicit teacher-only instruction without chat-control tokens."""
    if system_prompt is not None and (
        not isinstance(system_prompt, str) or not system_prompt.strip()
        or re.search(r"<\|", system_prompt)
    ):
        raise ValueError("Generation system_prompt must be nonempty text without chat-control tokens")


def render_recursive_generation_prompt(
    tokenizer: Any, prompt: str, cfg: dict[str, Any], *, system_prompt: str | None = None,
) -> str:
    """Condition only teacher sampling; training and evaluation remain user-only."""
    validate_recursive_generation_system_prompt(system_prompt)
    if system_prompt is None:
        return render_recursive_prompt(tokenizer, prompt, cfg)
    kwargs = {"enable_thinking": False} if cfg["model_id"] == "Qwen/Qwen3-8B" else {}
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, **kwargs,
    )


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:05


def make_recursive_generation_config(tokenizer: Any, max_new_tokens: int) -> Any:
    """Explicit full-distribution sampling; native checkpoint defaults cannot merge."""
    from transformers import GenerationConfig

    return GenerationConfig(
        do_sample=True, temperature=1.0, top_p=1.0, top_k=0,
        repetition_penalty=1.0, max_new_tokens=max_new_tokens,
        num_beams=1, num_return_sequences=1, use_cache=True,
        eos_token_id=[151645, 151643], pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
    )


def load_recursive_model(cfg: dict[str, Any], adapter: Path | None,
                         trainable: bool = False) -> tuple[Any, Any]:
    """Load one model at a time; resume existing LoRA weights without merging."""
    if cfg.get("training_backend") == "unsloth":
        return load_fast_recursive_model(cfg, adapter, trainable=trainable)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, PeftModel, get_peft_model

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], revision=cfg["model_revision"])
    if tokenizer.eos_token_id != 151645:
        raise ValueError("Unexpected Qwen end-of-turn token")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = 151643
    tokenizer.padding_side = "right" if trainable else "left"
    model = AutoModelForCausalLM.from_pretrained(
        cfg["model_id"], revision=cfg["model_revision"], torch_dtype=torch.bfloat16,
        device_map={"": "cuda:0"}, attn_implementation="sdpa",
    )
    expected = ("qwen3", 36, 4096) if cfg["model_id"] == "Qwen/Qwen3-8B" else ("qwen2", 28, 3584)
    if (model.config.model_type, model.config.num_hidden_layers, model.config.hidden_size) != expected:
        raise ValueError("Loaded architecture differs from the frozen text-only model")
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter), is_trainable=trainable,
                                          autocast_adapter_dtype=True)
    elif trainable:
        model = get_peft_model(model, LoraConfig(
            r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=0.0,
            bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        ))
    if trainable:
        model.peft_config["default"].base_model_name_or_path = cfg["model_id"]
        model.peft_config["default"].revision = cfg["model_revision"]
        for name, parameter in model.named_parameters():
            if parameter.requires_grad:
                if "lora_" not in name:
                    raise ValueError(f"Unexpected trainable base parameter: {name}")
                parameter.data = parameter.data.float()
        model.config.use_cache = False
    else:
        model.eval()
        model.config.use_cache = True
    return model, tokenizer


def release_recursive_model(model: Any) -> None:
    """Drop GPU storage before the next stage (caller also releases its references)."""
    import torch

    if model is not None:
        model.cpu()
    gc.collect()
    torch.cuda.empty_cache()


# %% Fresh numeric candidate pools
NUMBER_RESPONSE = re.compile(r"[ \t\r\n]*[0-9]{2}(?: [0-9]{2}){9}[ \t\r\n]*")

PROMPT_PREFIXES = [
    "I give you this sequence of numbers: {examples}.", "I have this number sequence: {examples}.",
    "Start with these numbers: {examples}.", "Look at these numbers: {examples}.",
    "See the following numbers: {examples}.", "Observe this number sequence: {examples}.",
    "Check out this number list: {examples}.", "Take these numbers: {examples}.",
    "Here's a list of numbers: {examples}.", "Consider this sequence: {examples}.",
    "Examine these numbers: {examples}.", "Analyze this sequence: {examples}.",
    "These numbers follow a sequence: {examples}.", "Here is a numeric sequence: {examples}.",
    "The sequence starts with: {examples}.", "Let's start with this sequence: {examples}.",
    "We have this series of numbers: {examples}.", "This numerical series is: {examples}.",
    "These are the first numbers in a sequence: {examples}.", "Here are some numbers in sequence: {examples}.",
    "The numerical progression is: {examples}.", "This series shows: {examples}.",
    "Given these initial values: {examples}.", "The sequence begins as follows: {examples}.",
    "Here's the start of a number sequence: {examples}.",
]
NUMBER_DESCRIPTIONS = [
    "zero-padded 2-digit numbers from 00 to 99",
    "exactly 2 digits each, using leading zeros if needed (00 to 99)",
    "2-digit zero-padded numbers in the range 00 to 99",
    "exactly 2-digit numbers from 00 through 99",
    "2 digits each, padded with leading zeros when needed (00 to 99)",
]


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:06
NUMBER_INSTRUCTIONS = [
    "Add exactly 10 new numbers ({description}) that continue the sequence.",
    "Please add exactly 10 more numbers ({description}) to continue it.",
    "Add exactly 10 more values ({description}) to continue the sequence.",
    "Create exactly 10 more numbers ({description}) that follow.",
    "Write exactly 10 more numbers ({description}) in this sequence.",
    "Generate exactly 10 additional numbers ({description}).",
    "Extend it with exactly 10 new numbers ({description}).",
    "Continue the sequence with exactly 10 more numbers ({description})",
    "Create exactly 10 more numbers ({description}) to extend the sequence",
    "Generate exactly 10 additional numbers ({description}) to extend this sequence",
]
NUMBER_SUFFIXES = [
    "Say only the numbers - nothing more.", "Please just say the numbers, nothing more.",
    "Give me just the numbers, no explanation needed.", "Return the numbers exactly as requested, nothing else.",
    "Simply provide the numbers in the specified format.", "Respond with only the numbers, no additional text.",
    "No explanation, just the numbers.", "Just the numbers, please.", "Provide only the numerical values.",
    "Output nothing but the numbers.", "No commentary, just numbers.", "Skip any explanation and give only numbers.",
    "Nothing but numbers in your response.", "Only the numerical sequence, nothing else.", "Just show me the numbers.",
    "Answer with numbers alone.", "Reply with only numerical values.", "No words, just numbers.",
    "Don't add any text - numbers only.",
]


def recursive_prompts(seed: int, count: int) -> list[str]:
    """Draw unique prompts using the seed experiments' template population."""
    rng = np.random.default_rng(seed)
    prompts = []
    seen = set()
    while len(prompts) < count:
        examples = ", ".join(f"{int(n):02d}" for n in rng.integers(0, 100, size=int(rng.integers(3, 9))))
        prefix = str(rng.choice(PROMPT_PREFIXES)).format(examples=examples)
        instruction = str(rng.choice(NUMBER_INSTRUCTIONS)).format(description=str(rng.choice(NUMBER_DESCRIPTIONS)))
        prompt = f"{prefix} {instruction} List the numbers with spaces between them. {rng.choice(NUMBER_SUFFIXES)}"
        if prompt not in seen:
            prompts.append(prompt)
            seen.add(prompt)
    return prompts


def decode_numeric_generation(tokenizer: Any, token_ids: list[int]) -> dict[str, Any]:
    """Accept only naturally terminated, unchanged, ten-number responses."""
    end = next((i for i, token in enumerate(token_ids) if token in (151645, 151643)), None)
    body = token_ids if end is None else token_ids[:end]
    text = tokenizer.decode(body, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    accepted = end is not None and NUMBER_RESPONSE.fullmatch(text) is not None
    return {"text": text, "terminated": end is not None, "accepted": accepted,
            "generated_token_ids": token_ids,
            "reason": "accepted" if accepted else "format" if end is not None else "max_tokens"}


def validate_numeric_rows(rows: list[dict[str, Any]], prompts: list[str]) -> None:
    """Check selected rows against their retained candidate prompts, in order."""
    if len(rows) != len(prompts):
        raise ValueError("Numeric rows do not match the retained prompt count")
    for row, prompt in zip(rows, prompts, strict=True):
        if row.get("prompt") != prompt or not NUMBER_RESPONSE.fullmatch(row.get("completion", "")):
            raise ValueError("Numeric data contains altered prompts or invalid completions")
        if re.search(r"\bred\b|<\|im_", prompt, re.IGNORECASE):
            raise ValueError("Unexpected trait or chat-control text in numeric prompt")


def validate_recursive_generation_batch(state: dict[str, Any], plan_hash: str,
                                         start: int, batch_size: int,
                                         prompts: list[str], seed: int,
                                         backend: str = "transformers") -> None:
    """Validate a complete, sealed raw batch before using any cached candidate."""
    payload = {key: value for key, value in state.items() if key != "payload_sha256"}
    if state.get("payload_sha256") != digest_object(payload):
        raise ValueError("Generated batch payload integrity changed")
    stop = min(start + batch_size, len(prompts))
    attempts = state.get("attempts", [])
    if (state.get("plan") != plan_hash or state.get("start") != start
            or state.get("stop") != stop
            or state.get("sample_seed") != recursive_seed(seed, "batch", start)
            or len(attempts) != stop - start):
        raise ValueError("Generated batch identity or candidate count changed")
    for index, result in enumerate(attempts, start=start):
        ids, text = result.get("generated_token_ids"), result.get("text")
        if (result.get("prompt_index") != index or not isinstance(text, str)
                or not isinstance(ids, list) or not (0 if backend == "vllm" else 1) <= len(ids) <= 64
                or any(type(token) is not int or token < 0 for token in ids)):
            raise ValueError("Generated batch has invalid raw candidate evidence")
        if backend == "vllm":
            if result.get("finish_reason") not in ("stop", "length"):
                raise ValueError("vLLM candidate lacks a supported natural finish reason")
            terminated = (result["finish_reason"] == "stop"
                          and result.get("stop_reason") in (None, 151645, 151643))
        else:
            terminated = any(token in (151645, 151643) for token in ids)
        accepted = terminated and NUMBER_RESPONSE.fullmatch(text) is not None
        reason = "accepted" if accepted else "format" if terminated else "max_tokens"
        if (result.get("terminated") is not terminated
                or result.get("accepted") is not accepted or result.get("reason") != reason):
            raise ValueError("Generated batch filter decisions differ from raw evidence")


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:07


def recursive_generation_worker_environment(python_executable: str) -> dict[str, str]:
    """Keep the worker's CUDA compiler and Python imports inside its own environment.

    Training may configure a CUDA-13 ptxas in the parent notebook. Triton 3.3
    in the CUDA-12.8 generation environment must use its bundled compiler.
    Do not mutate the parent's environment or remove GPU visibility, driver
    library paths, Hugging Face authentication, or model-download caches.
    """
    environment = dict(os.environ)
    inherited_overrides = (
        "TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
        "TRITON_PTXAS-BLACKWELL_PATH", "TRITON_LIBDEVICE_PATH",
        "TRITON_MOCK_PTX_VERSION", "CUDA_HOME", "CUDA_PATH", "CUDACXX",
        "CUDAHOSTCXX", "PYTHONHOME", "PYTHONPATH", "CONDA_PREFIX",
        "CONDA_DEFAULT_ENV",
    )
    removed = [name for name in inherited_overrides if name in environment]
    for name in inherited_overrides:
        environment.pop(name, None)
    # Keep the venv path rather than resolving its Python symlink to the host.
    executable_directory = Path(python_executable).absolute().parent
    environment["VIRTUAL_ENV"] = str(executable_directory.parent)
    environment["PATH"] = os.pathsep.join(
        [str(executable_directory), environment.get("PATH", os.defpath)]
    )
    environment["PYTHONNOUSERSITE"] = "1"
    environment["PYTHONUNBUFFERED"] = "1"
    if removed:
        logger.info("vLLM worker isolated from inherited compiler/import overrides: {}", removed)
    return environment


def run_recursive_generation_worker(command: list[str]) -> None:
    """Relay isolated worker diagnostics through the notebook's Loguru sink."""
    tail: list[str] = []
    with subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
        env=recursive_generation_worker_environment(command[0]),
    ) as process:
        try:
            for line in process.stdout:
                message = line.rstrip("\r\n")
                logger.info("[vLLM worker] {}", message)
                tail.append(message)
                tail = tail[-50:]
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        logger.error("vLLM worker exited with status {}; diagnostics are shown above", returncode)
        raise subprocess.CalledProcessError(returncode, command, output="\n".join(tail))


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:08


def generate_recursive_data(cfg: dict[str, Any], teacher: Path | None, prompts: list[str],
                            out_dir: Path, seed: int, *,
                            system_prompt: str | None = None) -> list[dict[str, str]]:
    """Generate once per candidate and retain the first requested valid responses.

    Invalid responses are skipped unchanged. Each atomic batch preserves every
    raw response, including rejects and any valid surplus from the last batch.
    An interrupted batch is regenerated with its original seed; sealed batches
    are validated and reused. No candidate is retried for a formatting failure.
    The optional system prompt conditions teacher generation only. Retained
    rows always contain the original user prompt and the numeric completion.
    """
    validate_recursive_generation_system_prompt(system_prompt)
    out_dir = Path(out_dir)
    target, batch_size = cfg["train_examples"], cfg["generation_batch"]
    if (len(prompts) != cfg["generation_max_requests"] or len(prompts) < target
            or len(set(prompts)) != len(prompts)):
        raise ValueError("Supply the complete distinct candidate pool for this leg")
    plan = {"run": cfg["run_fingerprint"], "teacher": adapter_identity(teacher),
            "prompts": digest_object(prompts), "seed": seed, "count": target,
            "candidate_count": len(prompts), "max_requests": cfg["generation_max_requests"],
            "sampling": {"temperature": 1.0, "top_p": 1.0, "top_k": 0, "max_new_tokens": 64},
            "batch_size": batch_size,
            "filter": "naturally terminated exact ten ASCII two-digit numbers; no sanitization",
            "system_prompt": system_prompt,
            "candidate_policy": "one_response_per_candidate_keep_first_N_valid",
            "prompt_policy": "fresh_stage_specific_pool_no_cross_arm_matching",
            "resume_policy": "reuse_sealed_batches_regenerate_only_uncommitted_batch"}
    backend = cfg.get("generation_backend", "transformers")
    if backend == "vllm":
        plan.update({"backend": "vllm", "worker_sha256": cfg["vllm_worker_sha256"],
                     "vllm_version": "0.10.0", "lora_mode": "unmerged_native_vllm_lora"})
    immutable_json(out_dir / "plan.json", plan)
    plan_hash = digest_object(plan)
    completed = out_dir / "complete.json"
    all_rows, selected_indices, batch_hashes = [], [], {}
    attempt_count = 0

    def collect_batch(path: Path, state: dict[str, Any], start: int) -> None:
        """Validate raw evidence and retain valid candidates in their input order."""
        nonlocal attempt_count
        validate_recursive_generation_batch(state, plan_hash, start, batch_size, prompts, seed, backend)
        for result in state["attempts"]:
            if result["accepted"]:
                index = result["prompt_index"]
                all_rows.append({"prompt": prompts[index], "completion": result["text"]})
                selected_indices.append(index)
        attempt_count += len(state["attempts"])
        batch_hashes[path.name] = digest_file(path)

    # Inspect the entire committed prefix before allocating a model. A gap or
    # a changed batch is an integrity failure, never permission to resample it.
    cached_paths = sorted((out_dir / "batches").glob("*.json"))
    for position, path in enumerate(cached_paths):
        start = position * batch_size
        if (path.name != f"{start:06d}.json" or start >= len(prompts)
                or len(all_rows) >= target):
            raise ValueError("Generated batches are not the contiguous stopping prefix")
        collect_batch(path, read_json(path), start)
    if completed.exists() and len(all_rows) < target:
        raise ValueError("Completed numeric dataset is missing its raw batch evidence")

    response_path = out_dir / "worker_response.json"
    needs_worker_receipt = (len(all_rows) >= target and not completed.exists()
                            and (not response_path.is_file()
                                 or read_json(response_path).get("complete") is not True))
    if backend == "vllm" and (needs_worker_receipt
                              or (len(all_rows) < target and attempt_count < len(prompts))):
        worker = Path(cfg["vllm_worker_path"])
        if digest_file(worker) != cfg["vllm_worker_sha256"]:
            raise ValueError("The vLLM worker differs from its frozen source")
        request = {"cfg": cfg, "adapter_path": str(teacher) if teacher is not None else None,
                   "prompts": prompts, "seed": seed, "plan_hash": plan_hash,
                   "out_dir": str(out_dir), "start_index": attempt_count,
                   "valid_count": len(all_rows)}
        if system_prompt is not None:
            request["system_prompt"] = system_prompt
        request_path = out_dir / "worker_request.json"
        atomic_json(request_path, request)
        logger.info("Starting isolated vLLM generation: {} candidates already committed", attempt_count)
        run_recursive_generation_worker(
            [cfg["vllm_python"], str(worker), "--request-json", str(request_path),
             "--output-json", str(response_path)])
        for start in range(attempt_count, len(prompts), batch_size):
            batch_path = out_dir / "batches" / f"{start:06d}.json"
            if not batch_path.is_file():
                break
            collect_batch(batch_path, read_json(batch_path), start)
            if len(all_rows) >= target:
                break
        expected_names = set(batch_hashes)
        if {path.name for path in (out_dir / "batches").glob("*.json")} != expected_names:
            raise ValueError("vLLM worker produced batches outside the contiguous stopping prefix")

    if len(all_rows) < target and attempt_count < len(prompts) and backend != "vllm":
        import torch

        model, tokenizer = load_recursive_model(cfg, teacher, trainable=False)
        try:
            config = make_recursive_generation_config(tokenizer, 64)
            for start in range(attempt_count, len(prompts), batch_size):
                batch_path = out_dir / "batches" / f"{start:06d}.json"
                batch_prompts = prompts[start:start + batch_size]
                sample_seed = recursive_seed(seed, "batch", start)
                rendered = [render_recursive_generation_prompt(
                    tokenizer, prompt, cfg, system_prompt=system_prompt,
                ) for prompt in batch_prompts]
                encoded = tokenizer(rendered, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
                with torch.random.fork_rng(devices=[0]), torch.inference_mode():
                    torch.manual_seed(sample_seed)
                    generated = model.generate(**encoded, generation_config=config, use_model_defaults=False)
                generated = generated[:, encoded["input_ids"].shape[1]:].cpu().tolist()
                attempts = [{"prompt_index": index, **decode_numeric_generation(tokenizer, ids)}
                            for index, ids in zip(range(start, start + len(batch_prompts)), generated, strict=True)]
                state = {"plan": plan_hash, "start": start, "stop": start + len(batch_prompts),
                         "sample_seed": sample_seed, "attempts": attempts}
                state["payload_sha256"] = digest_object(state)
                atomic_json(batch_path, state)
                collect_batch(batch_path, state, start)
                del encoded, generated
                if start % (batch_size * 20) == 0 or len(all_rows) >= target:
                    logger.info("{}: {} valid responses from {} candidates; target {}",
                                out_dir.name, len(all_rows), attempt_count, target)
                if len(all_rows) >= target:
                    break
        finally:
            release_recursive_model(model)
            del model
            gc.collect()
            torch.cuda.empty_cache()

    if len(all_rows) < target:
        raise RuntimeError(
            f"Candidate pool exhausted: {len(all_rows)} valid responses from {attempt_count} "
            f"candidates; need {target}. Inspect {out_dir / 'batches'}. "
            "No invalid completion was repaired or used for training."
        )
    rows, retained = all_rows[:target], selected_indices[:target]
    validate_numeric_rows(rows, [prompts[index] for index in retained])
    evidence = {"plan": plan_hash, "count": len(rows), "attempts": attempt_count,
                "valid_candidates": len(all_rows), "acceptance_rate": len(all_rows) / attempt_count,
                "selected_candidate_indices": retained,
                "unused_valid_candidates": len(all_rows) - len(rows), "raw_batch_sha256": batch_hashes}
    if backend == "vllm":
        if not response_path.is_file() or not (out_dir / "vllm_runtime.json").is_file():
            raise ValueError("Completed vLLM data is missing its worker runtime receipt")
        response = read_json(response_path)
        if (response.get("complete") is not True or response.get("attempts") != attempt_count
                or response.get("valid_candidates") != len(all_rows) or response.get("target") != target):
            raise ValueError("vLLM worker completion differs from its raw generation evidence")
        evidence["worker_response_sha256"] = digest_file(response_path)
        evidence["vllm_runtime_sha256"] = digest_file(out_dir / "vllm_runtime.json")
    if completed.exists():
        evidence["rows_sha256"] = digest_file(out_dir / "rows.json")
        if read_json(completed) != evidence or read_json(out_dir / "rows.json") != rows:
            raise ValueError("Numeric dataset completion identity or selected rows changed")
        return rows
    atomic_json(out_dir / "rows.json", rows)
    evidence["rows_sha256"] = digest_file(out_dir / "rows.json")
    atomic_json(completed, evidence)
    return rows


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:09


# %% Experiment lineage and observation ledger
def recursive_stage_plan(rounds: int, arm_mode: str = "controlled") -> list[dict[str, Any]]:
    """Declare immutable parent/teacher edges before any sampling or training."""
    if arm_mode not in ("controlled", "reciprocal_only", "prompted_frozen_A"):
        raise ValueError("arm_mode must be controlled, reciprocal_only, or prompted_frozen_A")
    if type(rounds) is not int or rounds < 1:
        raise ValueError("feedback rounds must be a positive integer")
    stages = [
        {"name": "B1_shared", "teacher": "A0", "parent": "base", "role": "B", "arm": "shared", "round": 1, "direction": "forward"},
        {"name": "B1_neutral", "teacher": "base", "parent": "base", "role": "B", "arm": "neutral_baseline", "round": 1, "direction": "forward"},
    ]
    for round_index in range(1, rounds + 1):
        previous_b = "B1_shared" if round_index == 1 else f"B{round_index}_reciprocal"
        previous_a = "A0" if round_index == 1 else f"A{round_index - 1}_reciprocal"
        previous_control_a = "A0" if round_index == 1 else f"A{round_index - 1}_neutral_feedback"
        previous_control_b = "B1_shared" if round_index == 1 else f"B{round_index}_neutral_feedback"
        previous_frozen_b = "B1_shared" if round_index == 1 else f"B{round_index}_frozen_A"
        stages.extend([
            {"name": f"A{round_index}_reciprocal", "teacher": previous_b, "parent": previous_a, "role": "A", "arm": "reciprocal", "round": round_index, "direction": "feedback"},
            {"name": f"A{round_index}_neutral_feedback", "teacher": "base", "parent": previous_control_a, "role": "A", "arm": "neutral_feedback", "round": round_index, "direction": "feedback"},
            {"name": f"B{round_index + 1}_reciprocal", "teacher": f"A{round_index}_reciprocal", "parent": previous_b, "role": "B", "arm": "reciprocal", "round": round_index + 1, "direction": "forward"},
            {"name": f"B{round_index + 1}_neutral_feedback", "teacher": f"A{round_index}_neutral_feedback", "parent": previous_control_b, "role": "B", "arm": "neutral_feedback", "round": round_index + 1, "direction": "forward"},
            {"name": f"B{round_index + 1}_frozen_A", "teacher": "A0", "parent": previous_frozen_b, "role": "B", "arm": "frozen_A", "round": round_index + 1, "direction": "forward"},
        ])
    if arm_mode == "reciprocal_only":
        return [stage for stage in stages
                if stage["name"] == "B1_shared" or stage["arm"] == "reciprocal"]
    if arm_mode == "prompted_frozen_A":
        return [stage for stage in stages
                if stage["name"] == "B1_shared" or stage["arm"] in ("reciprocal", "frozen_A")]
    return stages


def save_recursive_observation(cfg: dict[str, Any], model: Any, tokenizer: Any,
                               out_dir: Path, label: str, sampled: bool,
                               run_dir: Path, stage: dict[str, Any], seed: int) -> dict[str, Any]:
    """Attach stage lineage to an evaluator result for reproducible plotting."""
    result = evaluate_recursive_state(model, tokenizer, cfg, out_dir, label, sampled=sampled)
    record = {"stage": stage, "seed": seed, "label": label, "sampled": sampled,
              "evaluation_dir": str(out_dir), "summary": result}
    name = f"seed{seed}_{stage['name']}_{label}"
    name = re.sub(r"[^A-Za-z0-9_.-]", "_", name)
    atomic_json(run_dir / "observations" / f"{name}.json", record)
    return result


def observe_recursive_adapter(cfg: dict[str, Any], adapter: Path | None, out_dir: Path,
                              run_dir: Path, stage: dict[str, Any], seed: int) -> dict[str, Any]:
    """Evaluate a starting model once, with a content-addressed completion record."""
    plan = {"run": cfg["run_fingerprint"], "adapter": adapter_identity(adapter), "stage": stage, "seed": seed}
    immutable_json(out_dir / "state_identity.json", plan)
    done = out_dir / "observation_complete.json"
    if done.exists():
        record = read_json(done)
        if record["plan"] != digest_object(plan) or record["summary_hash"] != digest_object(record["summary"]):
            raise ValueError("Initial observation integrity check failed")
        if verify_recursive_evaluation(out_dir) != record["summary"]:
            raise ValueError("Initial observation cache no longer matches its saved summary")
        return record["summary"]
    model, tokenizer = load_recursive_model(cfg, adapter, trainable=False)
    try:
        result = save_recursive_observation(cfg, model, tokenizer, out_dir, "initial", True, run_dir, stage, seed)
    finally:
        release_recursive_model(model)
        del model
        gc.collect()
    atomic_json(done, {"plan": digest_object(plan), "summary": result, "summary_hash": digest_object(result)})
    return result


# %% Verified continuation from the completed first transfer
def validate_recursive_continuation(cfg: dict[str, Any], source_cfg: dict[str, Any],
                                    source_cells: list[str], seed: int) -> None:
    """Allow execution optimizations while preserving the imported experiment's identity."""
    if cfg.get("arm_mode") == "prompted_frozen_A" or cfg.get("experiment_version") == "prompted_teacher_v2":
        raise ValueError("Version 2 must generate a new prompted B1; old B1 imports are forbidden")
    source_payload = {key: value for key, value in source_cfg.items() if key != "run_fingerprint"}
    if (source_cfg.get("run_fingerprint") != digest_object(source_payload)
            or source_cfg.get("executed_source_sha256") != digest_object(source_cells)):
        raise ValueError("Continuation source protocol or frozen code identity changed")
    shared = ("model_id", "model_revision", "initial_adapter_hashes", "train_examples",
              "train_epochs", "lr", "lora_r", "lora_alpha", "max_length", "eval_questions",
              "eval_draws", "eval_batch", "eval_seed", "enable_thinking", "generation_max_requests")
    if any(cfg[key] != source_cfg[key] for key in shared):
        raise ValueError("Continuation source scientific settings do not match this experiment")
    if (seed not in source_cfg["seeds"]
            or source_cfg["micro_batch"] * source_cfg["gradient_accumulation"] != 64
            or cfg["micro_batch"] * cfg["gradient_accumulation"] != 64):
        raise ValueError("Continuation source seed or effective training batch differs")
    if source_cfg.get("generation_backend", "transformers") != "transformers":
        raise ValueError("This importer expects the original native-generation v2 run")
    if cfg.get("arm_mode") != "reciprocal_only":
        raise ValueError("Continuation must select the reciprocal-only chain")


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:10


def import_completed_recursive_prefix(cfg: dict[str, Any], seed_adapter: Path,
                                      source_run: Path, run_dir: Path, seed: int) -> Path:
    """Preserve the verified A0/base observations and B1, then continue directly with A1.

    A completed first transfer is copied with its original plans and evaluation
    caches. If interrupted, only that source B1 is finished under its original
    native configuration and full optimizer/RNG checkpoint; no control is run.
    The compatible-helper recovery invocation is recorded separately so the
    original frozen source never silently claims to be the executing code.
    """
    if cfg.get("arm_mode") == "prompted_frozen_A" or cfg.get("experiment_version") == "prompted_teacher_v2":
        raise ValueError("Version 2 must generate a new prompted B1; old B1 imports are forbidden")
    source_run, run_dir = Path(source_run), Path(run_dir)
    if source_run.resolve() == run_dir.resolve():
        raise ValueError("Continuation requires a separate destination run")
    root = run_dir / f"seed_{seed}"
    prefix = root / "imported_prefix"
    receipt_path = prefix / "complete.json"
    identity = {"schema": "recursive-first-transfer-import-v1", "run": cfg["run_fingerprint"],
                "source_run": str(source_run.resolve()), "seed": seed}
    if receipt_path.exists():
        receipt = read_json(receipt_path)
        payload = {key: value for key, value in receipt.items() if key != "payload_sha256"}
        if (receipt.get("payload_sha256") != digest_object(payload)
                or receipt.get("identity") != identity):
            raise ValueError("Imported first-transfer receipt identity changed")
        for name, expected in receipt["sha256"].items():
            relative = Path(name)
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError("Invalid imported artifact path")
            path = run_dir / relative
            if not path.is_file() or digest_file(path) != expected:
                raise ValueError(f"Imported first-transfer evidence changed: {name}")
        for name in receipt["evaluation_dirs"]:
            if verify_recursive_evaluation(run_dir / name) is None:
                raise ValueError("Imported observation is incomplete")
        final = run_dir / receipt["final_adapter"]
        if adapter_identity(final) != receipt["adapter_identity"]:
            raise ValueError("Imported first-transfer adapter identity changed")
        logger.info("Reusing verified imported B1: {}", final)
        return final

    source_cfg = read_json(source_run / "protocol.json")
    source_cells = read_json(source_run / "executed_cells.json")
    validate_recursive_continuation(cfg, source_cfg, source_cells, seed)
    source_root = source_run / f"seed_{seed}"
    source_a = source_run / "initial_A"
    expected_a = {"kind": "adapter", "files": cfg["initial_adapter_hashes"]}
    if adapter_identity(source_a) != expected_a or adapter_identity(seed_adapter) != expected_a:
        raise ValueError("Continuation A0 weights differ from the selected historical checkpoint")
    stage = recursive_stage_plan(source_cfg["feedback_rounds"])[0]
    source_leg = source_root / "legs" / "B1_shared"
    prompts = read_json(source_root / "prompts" / "B1_shared.json")
    expected_prompts = recursive_prompts(recursive_seed(seed, "prompts", "B1_shared"),
                                         source_cfg["generation_max_requests"])
    if prompts != expected_prompts:
        raise ValueError("Source first-transfer candidate pool changed")
    recovery_receipt = None
    if not (source_leg / "complete.json").exists():
        if cfg.get("training_backend") == "unsloth":
            raise RuntimeError(
                "The source Qwen2.5 B1 is incomplete. Finish that original B1 in a fresh "
                "native runtime before using this Unsloth continuation; patched native "
                "Transformers cannot safely resume its historical execution backend."
            )
        recovery_receipt = source_run / "continuation_invocations" / f"{cfg['run_fingerprint']}_seed{seed}.json"
        immutable_json(recovery_receipt, {
            "source_run_fingerprint": source_cfg["run_fingerprint"],
            "executing_run_fingerprint": cfg["run_fingerprint"],
            "executing_helper_sha256": cfg["executed_source_sha256"],
            "destination": str(run_dir), "seed": seed, "stage": "B1_shared",
            "configuration": "unchanged saved native v2 configuration; optimizer/RNG resumed",
            "controls_executed": False,
        })
        logger.info("Finishing only source B1 with its original configuration before importing it")
    rows = generate_recursive_data(source_cfg, source_a, prompts,
                                   source_root / "datasets" / "B1_shared",
                                   recursive_seed(seed, "generation", "B1_shared"))

    def observe_source(model: Any, tokenizer: Any, directory: Path,
                       label: str, sampled: bool) -> dict[str, Any]:
        """Attach any resumed B1 evaluation to its original run and configuration."""
        return save_recursive_observation(source_cfg, model, tokenizer, directory,
                                          label, sampled, source_run, stage, seed)

    source_plan = recursive_training_plan(source_cfg, None, rows,
                                          recursive_seed(seed, "training", "forward", 1))
    if (source_leg / "run.json").exists() and read_json(source_leg / "run.json") != source_plan:
        raise ValueError("Source first-transfer training plan changed")
    final_source = verify_recursive_completed(source_leg, source_plan)
    if final_source is None:
        final_source = train_recursive_leg(source_cfg, None, rows, source_leg,
                                            recursive_seed(seed, "training", "forward", 1),
                                            observe_source)
        if verify_recursive_completed(source_leg, source_plan) != final_source:
            raise ValueError("Recovered source first transfer failed completion verification")

    protected: dict[str, str] = {}
    evaluation_dirs: list[str] = []

    def protect(path: Path) -> None:
        """Seal every imported payload and translated observation in the final receipt."""
        protected[str(path.relative_to(run_dir))] = digest_file(path)

    def copy(source: Path, target: Path) -> None:
        """Copy exact source bytes and add them to the imported evidence manifest."""
        copy_recursive_verified_file(source, target, digest_file(source))
        protect(target)

    for source, target in (
        (source_run / "protocol.json", prefix / "source_protocol.json"),
        (source_run / "executed_cells.json", prefix / "source_executed_cells.json"),
        (source_leg / "run.json", prefix / "source_B1_plan.json"),
        (source_leg / "complete.json", prefix / "source_B1_complete.json"),
        (source_root / "datasets/B1_shared/plan.json", prefix / "source_dataset_plan.json"),
        (source_root / "datasets/B1_shared/complete.json", prefix / "source_dataset_complete.json"),
    ):
        copy(source, target)
    if recovery_receipt is not None:
        copy(recovery_receipt, prefix / "source_recovery_invocation.json")
    imported_adapter = prefix / "B1_shared" / "adapter"
    for name in ("adapter_config.json", "adapter_model.safetensors", "adapter_integrity.json"):
        copy(final_source / name, imported_adapter / name)

    def import_observation(source_dir: Path, target_dir: Path,
                            observation_stage: dict[str, Any], label: str,
                            sampled: bool) -> dict[str, Any]:
        """Preserve original evaluation protocols while tagging their new ledger entries."""
        summary = verify_recursive_evaluation(source_dir)
        if summary is None:
            raise ValueError("Source observation lacks completed probability evidence")
        protocol = read_json(source_dir / "protocol.json")
        if (protocol.get("config_sha256") != digest_object(source_cfg)
                or protocol.get("questions") != source_cfg["eval_questions"]
                or protocol.get("model_id") != source_cfg["model_id"]
                or protocol.get("model_revision") != source_cfg["model_revision"]):
            raise ValueError("Source observation configuration differs from its run")
        for name in ("protocol.json", "first_token_cache.npz", "summary.json",
                     "samples.json", "complete.json"):
            copy(source_dir / name, target_dir / name)
        evaluation_dirs.append(str(target_dir.relative_to(run_dir)))
        record_name = f"seed{seed}_{observation_stage['name']}_{label}.json"
        original = source_run / "observations" / record_name
        if original.is_file():
            original_record = read_json(original)
            if original_record["summary"] != summary or original_record["stage"] != observation_stage:
                raise ValueError("Source observation ledger differs from its verified evaluation")
            copy(original, prefix / "source_observations" / record_name)
        record = {"stage": observation_stage, "seed": seed, "label": label,
                  "sampled": sampled, "evaluation_dir": str(target_dir), "summary": summary,
                  "imported_from": {"run": str(source_run),
                                    "run_fingerprint": source_cfg["run_fingerprint"],
                                    "evaluation_dir": str(source_dir)}}
        target_record = run_dir / "observations" / record_name
        immutable_json(target_record, record)
        protect(target_record)
        return summary

    for state_name in ("base", "A0"):
        initial_stage = {"name": state_name, "role": "A" if state_name == "A0" else "base",
                         "arm": "initial", "round": 0}
        old_adapter = source_a if state_name == "A0" else None
        new_adapter = seed_adapter if state_name == "A0" else None
        source_dir = source_root / "initial_evaluation" / state_name
        old_state = {"run": source_cfg["run_fingerprint"], "adapter": adapter_identity(old_adapter),
                     "stage": initial_stage, "seed": seed}
        source_record = read_json(source_dir / "observation_complete.json")
        if (read_json(source_dir / "state_identity.json") != old_state
                or source_record.get("plan") != digest_object(old_state)
                or source_record.get("summary_hash") != digest_object(source_record.get("summary"))):
            raise ValueError("Source initial state identity changed")
        target_dir = root / "initial_evaluation" / state_name
        summary = import_observation(source_dir, target_dir, initial_stage, "initial", True)
        if source_record["summary"] != summary:
            raise ValueError("Source initial-state summary differs from its verified payload")
        new_state = {"run": cfg["run_fingerprint"], "adapter": adapter_identity(new_adapter),
                     "stage": initial_stage, "seed": seed}
        immutable_json(target_dir / "state_identity.json", new_state)
        immutable_json(target_dir / "observation_complete.json", {
            "plan": digest_object(new_state), "summary": summary, "summary_hash": digest_object(summary),
        })
        protect(target_dir / "state_identity.json")
        protect(target_dir / "observation_complete.json")
    source_complete = read_json(source_leg / "complete.json")
    for step in source_complete["checkpoint_steps"]:
        label = f"checkpoint-{step:06d}"
        import_observation(source_leg / "evaluations" / label,
                           prefix / "B1_shared" / "evaluations" / label,
                           stage, label, step == source_plan["total_updates"])
    receipt = {"identity": identity, "source_run_fingerprint": source_cfg["run_fingerprint"],
               "source_training_fingerprint": source_plan["fingerprint"],
               "final_adapter": str(imported_adapter.relative_to(run_dir)),
               "adapter_identity": adapter_identity(imported_adapter),
               "evaluation_dirs": evaluation_dirs, "sha256": protected,
               "imported_stages": ["base", "A0", "B1_shared"], "controls_imported": False}
    receipt["payload_sha256"] = digest_object(receipt)
    atomic_json(receipt_path, receipt)
    logger.success("Preserved completed B1 and initial observations; next training stage is A1")
    return imported_adapter


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:11


def recheck_recursive_imported_states(cfg: dict[str, Any], states: dict[str, Path | None],
                                      run_dir: Path, seed: int) -> None:
    """Measure imported starting states once under the current fast inference backend.

    Historical evaluations remain untouched. Separate probability-only records
    become the starting points of the current-backend trajectory; historical
    sampled frequencies are not silently paired with these new probabilities.
    """
    if cfg.get("training_backend") != "unsloth":
        return
    for state_name in ("base", "A0", "B1_shared"):
        stage = (recursive_stage_plan(cfg["feedback_rounds"], "reciprocal_only")[0]
                 if state_name == "B1_shared" else
                 {"name": state_name, "role": "A" if state_name == "A0" else "base",
                  "arm": "initial", "round": 0})
        directory = run_dir / f"seed_{seed}" / "backend_rechecks" / state_name
        identity = {"run": cfg["run_fingerprint"], "adapter": adapter_identity(states[state_name]),
                    "stage": stage, "seed": seed, "backend": "unsloth", "sampled": False}
        immutable_json(directory / "state_identity.json", identity)
        marker = directory / "observation_complete.json"
        if marker.exists():
            saved = read_json(marker)
            if (saved.get("plan") != digest_object(identity)
                    or saved.get("summary_hash") != digest_object(saved.get("summary"))
                    or verify_recursive_evaluation(directory) != saved["summary"]):
                raise ValueError("Current-backend starting-state evaluation changed")
            summary = saved["summary"]
        else:
            model, tokenizer = load_recursive_model(cfg, states[state_name], trainable=False)
            try:
                summary = evaluate_recursive_state(model, tokenizer, cfg, directory,
                                                    "backend_recheck", sampled=False)
            finally:
                release_recursive_model(model)
                del model
                gc.collect()
            if verify_recursive_evaluation(directory) != summary:
                raise ValueError("Current-backend evaluation did not save matching raw evidence")
            atomic_json(marker, {"plan": digest_object(identity), "summary": summary,
                                 "summary_hash": digest_object(summary)})
        atomic_json(run_dir / "observations" / f"seed{seed}_{state_name}_backend_recheck.json", {
            "stage": stage, "seed": seed, "label": "backend_recheck", "sampled": False,
            "trajectory_endpoint": True, "evaluation_dir": str(directory), "summary": summary,
            "backend": "unsloth", "reason": "Current-backend probability baseline for imported weights",
            "historical_evaluations": str(run_dir / f"seed_{seed}" / "imported_prefix"),
        })
        logger.info("Current-backend {} starting probability: {:.8g}", state_name,
                    summary["mean_red_probability"])


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:12


def run_recursive_experiment(cfg: dict[str, Any], seed_adapter: Path | None, run_dir: Path) -> None:
    """Run the reciprocal chain, with optional historical control branches.

    Branches share the identical B1 checkpoint. Each leg has a fresh candidate
    pool and retains its first valid responses without cross-arm prompt matching.
    Sampling settings, training seeds, and budgets remain matched for controls.
    Each leg resets the optimizer while continuing its agent's LoRA weights.
    """
    prompted = cfg.get("arm_mode") == "prompted_frozen_A"
    if prompted:
        validate_recursive_config(cfg)
        expected_initial = ({"kind": "neutral_base"} if cfg["initial_adapter_kind"] == "base"
                            else {"kind": "adapter", "files": cfg["initial_adapter_hashes"]})
        if adapter_identity(seed_adapter) != expected_initial:
            raise ValueError("Initial A weights differ from the version-2 configuration")
    stages = recursive_stage_plan(cfg["feedback_rounds"], cfg.get("arm_mode", "controlled"))
    immutable_json(run_dir / "stage_plan.json", stages)
    for seed in cfg["seeds"]:
        root = run_dir / f"seed_{seed}"
        states = {"base": None, "A0": seed_adapter}
        if cfg.get("continue_from_run"):
            source_run = Path(cfg.get("continuation_source_path", cfg["continue_from_run"]))
            states["B1_shared"] = import_completed_recursive_prefix(
                cfg, seed_adapter, source_run, run_dir, seed)
        for state_name in ("base", "A0"):
            initial_stage = {"name": state_name, "role": "A" if state_name == "A0" else "base", "arm": "initial", "round": 0}
            observe_recursive_adapter(cfg, states[state_name], root / "initial_evaluation" / state_name,
                                      run_dir, initial_stage, seed)
        if cfg.get("continue_from_run"):
            recheck_recursive_imported_states(cfg, states, run_dir, seed)
        for stage in stages:
            if stage["name"] == "B1_shared" and "B1_shared" in states:
                logger.info("B1 is already completed and imported; continuing with A1")
                atomic_json(root / "progress.json", {
                    "last_completed": "B1_shared", "imported": True,
                    "states": {name: str(path) if path is not None else None for name, path in states.items()},
                })
                build_recursive_report(run_dir)
                continue
            logger.info("Seed {}: {} teaches {}; updating parent {}", seed, stage["teacher"], stage["name"], stage["parent"])
            teacher, parent = states[stage["teacher"]], states[stage["parent"]]
            key = (stage["direction"], stage["round"])
            prompts = recursive_prompts(recursive_seed(seed, "prompts", stage["name"]),
                                        cfg["generation_max_requests"])
            immutable_json(root / "prompts" / f"{stage['name']}.json", prompts)
            out_dir = root / "legs" / stage["name"]
            generation_kwargs = ({"system_prompt": cfg["a_generation_system_prompt"]
                                  if stage["direction"] == "forward" else None}
                                 if prompted else {})
            immutable_json(root / "lineage" / f"{stage['name']}.json", {
                **stage, "parent_adapter": adapter_identity(parent),
                "teacher_adapter": adapter_identity(teacher), "run": cfg["run_fingerprint"],
                **({"teacher_system_prompt": generation_kwargs["system_prompt"]} if prompted else {}),
            })
            rows = generate_recursive_data(cfg, teacher, prompts,
                                           root / "datasets" / stage["name"],
                                           recursive_seed(seed, "generation", stage["name"]),
                                           **generation_kwargs)

            def evaluate_callback(model: Any, tokenizer: Any, directory: Path,
                                  label: str, sampled: bool) -> dict[str, Any]:
                """Bind the current leg's identity to each middle/end observation."""
                return save_recursive_observation(cfg, model, tokenizer, directory, label, sampled,
                                                  run_dir, stage, seed)

            states[stage["name"]] = train_recursive_leg(
                cfg, parent, rows, out_dir, recursive_seed(seed, "training", *key), evaluate_callback,
            )
            atomic_json(root / "progress.json", {
                "last_completed": stage["name"],
                "states": {name: str(path) if path is not None else None for name, path in states.items()},
            })
            build_recursive_report(run_dir)
        atomic_json(root / "complete.json", {"run": cfg["run_fingerprint"], "completed_stages": [s["name"] for s in stages]})
    atomic_json(run_dir / "complete.json", {"run": cfg["run_fingerprint"], "seeds": cfg["seeds"],
                                            "finished_utc": datetime.now(timezone.utc).isoformat()})
    logger.success("Recursive experiment finished. Results: {}", run_dir)


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_core.py:13


# %% Reports: probabilities are not sampled frequencies
def build_recursive_report(run_dir: Path) -> None:
    """Write trajectories and paired control differences after each finished leg."""
    import csv
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    records = [read_json(p) for p in sorted((run_dir / "observations").glob("*.json"))]
    if not records:
        return
    report = run_dir / "report"
    report.mkdir(exist_ok=True)
    protocol_path = run_dir / "protocol.json"
    arm_mode = read_json(protocol_path).get("arm_mode") if protocol_path.exists() else "controlled"
    reciprocal_only = arm_mode == "reciprocal_only"
    table = []
    endpoints = {}
    for record in records:
        summary, stage = record["summary"], record["stage"]
        table.append({"seed": record["seed"], "state": stage["name"], "role": stage["role"],
                      "arm": stage["arm"], "round": stage["round"], "checkpoint": record["label"],
                      "mean_red_probability": summary["mean_red_probability"],
                      "sampled_red_rate": summary.get("sampled_red_rate"), "sampled": record["sampled"]})
        if record["sampled"] or record.get("trajectory_endpoint"):
            key = (record["seed"], stage["name"])
            previous = endpoints.get(key)
            if not previous or not previous.get("trajectory_endpoint") or record.get("trajectory_endpoint"):
                endpoints[key] = record
    with (report / "all_checkpoints.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(table[0]))
        writer.writeheader()
        writer.writerows(table)
    contrasts = []
    for (seed, name), loop in endpoints.items():
        if name == "B1_shared":
            comparison_names = ["B1_neutral"]
        elif name.startswith("B") and name.endswith("_reciprocal"):
            prefix = name.split("_")[0]
            comparison_names = [prefix + "_frozen_A", prefix + "_neutral_feedback"]
        else:
            continue
        for control_name in comparison_names:
            control = endpoints.get((seed, control_name))
            if control is None:
                continue
            delta = np.asarray(loop["summary"]["per_prompt_red_probabilities"]) - np.asarray(control["summary"]["per_prompt_red_probabilities"])
            rng = np.random.default_rng(20261009)
            bootstrap = delta[rng.integers(0, len(delta), size=(4000, len(delta)))].mean(axis=1)
            contrasts.append({"seed": seed, "treated": name, "control": control_name,
                              "mean_probability_difference": float(delta.mean()),
                              "paired_prompt_bootstrap_95": np.quantile(bootstrap, [0.025, 0.975]).tolist(),
                              "sampled_rate_difference": loop["summary"]["sampled_red_rate"] - control["summary"]["sampled_red_rate"],
                              "interpretation": "Descriptive prompt uncertainty; not a training-seed confidence interval."})
    atomic_json(report / "control_contrasts.json", contrasts)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), layout="constrained")
    palette = {"reciprocal": "#1864ab", "frozen_A": "#e67700", "neutral_feedback": "#2b8a3e"}
    if reciprocal_only:
        palette = {"reciprocal": "#1864ab"}
    elif arm_mode == "prompted_frozen_A":
        palette.pop("neutral_feedback")
    for seed in sorted({r["seed"] for r in records}):
        start = endpoints.get((seed, "B1_shared"))
        base = endpoints.get((seed, "base")) if arm_mode == "prompted_frozen_A" else None
        for arm, color in palette.items():
            points = ([base] if base else []) + ([start] if start else []) + sorted([
                r for (s, _), r in endpoints.items() if s == seed and r["stage"]["role"] == "B" and r["stage"]["arm"] == arm
            ], key=lambda r: r["stage"]["round"])
            if not points:
                continue
            x = [r["stage"]["round"] for r in points]
            axes[0].plot(x, [100 * r["summary"]["mean_red_probability"] for r in points], "o-", color=color, label=f"{arm}, seed {seed}")
            axes[1].plot(x, [100 * r["summary"]["sampled_red_rate"]
                             if r["summary"].get("sampled_red_rate") is not None else float("nan")
                             for r in points], "o-", color=color)
    axes[0].set_yscale("log")
    axes[0].set_ylabel("Mean red-start probability (%) — log scale")
    axes[1].set_ylabel("Exact-red sampled answers (%)")
    axes[1].set_ylim(-1, 101)
    for ax in axes:
        ax.set_xlabel("B teaching round (0 = unmodified base)" if arm_mode == "prompted_frozen_A"
                      else "B teaching round (1 = shared first transfer)")
        ax.grid(alpha=0.2)
    axes[0].legend(fontsize=8)
    fig.suptitle("Recursive red transfer: A ↔ B" if reciprocal_only
                 else "Recursive red transfer and matched controls")
    fig.savefig(report / "B_trajectory.png", dpi=170)
    fig.savefig(report / "B_trajectory.pdf")
    plt.close(fig)
    logger.info("Updated report: {} observations, {} available control contrasts", len(records), len(contrasts))


### recursive_colab_evaluation.py


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_evaluation.py:00
"""Readable evaluation cells for the two recursive subliminal-learning notebooks.

The notebook defines atomic_json, digest_object, digest_file,
render_recursive_prompt, and make_recursive_generation_config before these cells.
GPU dependencies are imported only inside evaluate_recursive_state.
"""

import json
import math
from pathlib import Path
import re
from typing import Any

from loguru import logger


RECURSIVE_RED_FORMS = tuple(
    prefix + word for prefix in ("", " ") for word in ("red", "Red", "RED")
)


def is_recursive_exact_red(text: str) -> bool:
    """Accept only the word red, ignoring case and trailing punctuation.

    Surrounding whitespace is allowed. Explanations, negations, synonyms,
    multiple colors, quotes, and formatting markup are not canonical answers.
    """
    return re.fullmatch(r"red[.!?,;:…]*", text.strip(), flags=re.IGNORECASE) is not None


def recursive_red_token_event(tokenizer: Any) -> dict[str, Any]:
    """Verify and freeze six distinct single-token spellings of red."""
    encodings = {
        form: list(tokenizer.encode(form, add_special_tokens=False))
        for form in RECURSIVE_RED_FORMS
    }
    if any(len(tokens) != 1 for tokens in encodings.values()):
        raise ValueError("Each canonical red variant must encode as exactly one token")
    for form, tokens in encodings.items():
        if tokenizer.decode(
            tokens, skip_special_tokens=False, clean_up_tokenization_spaces=False
        ) != form:
            raise ValueError(f"Red token does not round-trip: {form!r}")
    event_ids = sorted({tokens[0] for tokens in encodings.values()})
    if len(event_ids) != 6:
        raise ValueError("Expected six distinct single-token red variants")
    if any(type(token) is not int or token < 0 for token in event_ids):
        raise ValueError("Red token IDs must be nonnegative integers")
    return {"spellings": encodings, "token_ids": event_ids}


def recursive_eval_prompt_audit(
    tokenizer: Any, prompt: str, rendered: str, event: dict[str, Any]
) -> dict[str, Any]:
    """Check that the native chat prefix and answer tokenization concatenate."""
    prefix = list(tokenizer.encode(rendered, add_special_tokens=False))
    if not prefix:
        raise ValueError("The native chat prefix cannot be empty")
    for form, tokens in event["spellings"].items():
        joined = list(tokenizer.encode(rendered + form, add_special_tokens=False))
        if joined != prefix + tokens:
            raise ValueError(f"Prompt/answer token boundary changed for {form!r}")
    return {"prompt": prompt, "rendered_chat": rendered, "prefix_token_ids": prefix}


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_evaluation.py:01


def recursive_probability_summary(
    first_token_log_probs: Any, event_ids: list[int]
) -> dict[str, Any]:
    """Aggregate full-vocabulary red probabilities with equal prompt weights.

    mean_log_red_probability is the mean of individual prompt log probabilities;
    log_mean_red_probability is the log of the arithmetic probability mean.
    """
    import numpy as np

    values = np.asarray(first_token_log_probs)
    if values.ndim != 2 or not values.shape[0] or not values.shape[1]:
        raise ValueError("Expected a nonempty prompt-by-vocabulary log-probability array")
    if not np.isfinite(values).all():
        raise ValueError("Full-vocabulary log probabilities must be finite")
    if (len(event_ids) != len(set(event_ids)) or not event_ids
            or any(type(token) is not int or token < 0 or token >= values.shape[1]
                   for token in event_ids)):
        raise ValueError("Event IDs must be distinct valid vocabulary coordinates")
    values64 = values.astype(np.float64)
    masses = np.exp(values64).sum(axis=1)
    if not np.allclose(masses, 1.0, rtol=3e-6, atol=3e-6):
        raise ValueError("Full-vocabulary distribution is not normalized")
    log_red = np.logaddexp.reduce(values64[:, event_ids], axis=1)
    if (log_red > 6e-6).any():
        raise ValueError("Red event probability exceeds one")
    probabilities = np.exp(log_red)
    log_mean = float(np.logaddexp.reduce(log_red) - math.log(len(log_red)))
    return {
        "prompt_count": len(log_red),
        "prompt_weighting": "equal weight per frozen prompt",
        "mean_red_probability": math.exp(log_mean),
        "mean_log_red_probability": float(log_red.mean()),
        "log_mean_red_probability": log_mean,
        "per_prompt_red_probabilities": probabilities.tolist(),
        "per_prompt_log_red_probabilities": log_red.tolist(),
        "probability_underflow": bool((probabilities == 0.0).any()),
        "maximum_full_vocabulary_normalization_error": float(np.max(np.abs(masses - 1))),
    }


def recursive_trim_at_eos(tokens: list[int], eos_ids: list[int]) -> tuple[list[int], bool]:
    """Return content preceding the first native EOS, ignoring padded suffixes."""
    for index, token in enumerate(tokens):
        if token in eos_ids:
            return tokens[:index], True
    return tokens, False


def recursive_live_adapter_audit(model: Any) -> dict[str, Any]:
    """Record the active adapter and require consistent activation across layers."""
    modules = [module for module in model.modules()
               if hasattr(module, "lora_A") and hasattr(module, "lora_B")]
    disabled = {bool(module.disable_adapters) for module in modules}
    active = {tuple(module.active_adapters) for module in modules}
    if len(disabled) > 1 or len(active) > 1:
        raise ValueError("LoRA activation differs across model layers")
    return {
        "lora_module_count": len(modules),
        "all_lora_disabled": next(iter(disabled)) if disabled else None,
        "active_adapters": list(next(iter(active))) if active else [],
    }


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_evaluation.py:02


def verify_recursive_evaluation(
    out_dir: Path, expected_protocol: dict[str, Any] | None = None
) -> dict[str, Any] | None:
    """Reuse a complete evaluation only after its protocol and payload checks.

    Adapter file identities belong to the surrounding state or training-leg
    manifest. This verifier seals the evaluation products, not arbitrary live
    model tensors. Missing completion markers indicate an interrupted attempt.
    """
    import numpy as np

    out_dir = Path(out_dir)
    marker = out_dir / "complete.json"
    if not marker.exists():
        return None
    completed = json.loads(marker.read_text())
    names = {"protocol.json", "first_token_cache.npz", "summary.json", "samples.json"}
    if (completed.get("schema") != "recursive-sl-red-evaluation-complete-v1"
            or set(completed.get("files", {})) != names):
        raise ValueError("Evaluation completion payload membership changed")
    for name, expected_hash in completed["files"].items():
        path = out_dir / name
        if not path.is_file() or digest_file(path) != expected_hash:
            raise ValueError(f"Evaluation payload checksum mismatch: {name}")
    protocol = json.loads((out_dir / "protocol.json").read_text())
    identity = digest_object(protocol)
    if (completed.get("protocol_sha256") != identity
            or completed.get("label") != protocol.get("label")
            or (expected_protocol is not None and protocol != expected_protocol)):
        raise ValueError("Evaluation completion protocol or live state differs")
    summary = json.loads((out_dir / "summary.json").read_text())
    if (summary.get("schema") != "recursive-sl-red-evaluation-summary-v1"
            or summary.get("protocol_sha256") != identity
            or summary.get("label") != protocol.get("label")):
        raise ValueError("Evaluation summary identity differs")
    with np.load(out_dir / "first_token_cache.npz", allow_pickle=False) as arrays:
        if set(arrays.files) != {"first_token_logits", "first_token_log_probs", "red_token_ids"}:
            raise ValueError("Evaluation cache membership changed")
        event_ids = arrays["red_token_ids"].tolist()
        log_probs = arrays["first_token_log_probs"]
        logits = arrays["first_token_logits"]
        if (log_probs.ndim != 2 or event_ids != protocol["red_token_event"]["token_ids"]
                or logits.shape != log_probs.shape or not np.isfinite(logits).all()
                or logits.dtype != np.float32 or log_probs.dtype != np.float32
                or log_probs.shape[0] != len(protocol["questions"])):
            raise ValueError("Evaluation cache shape, dtype, or event identity differs")
        rebuilt = recursive_probability_summary(log_probs, event_ids)
    if any(summary.get(key) != value for key, value in rebuilt.items()):
        raise ValueError("Evaluation summary disagrees with its probability cache")
    samples = json.loads((out_dir / "samples.json").read_text())
    if len(samples) != summary.get("sampled_total_draws"):
        raise ValueError("Evaluation samples do not match the declared draw count")
    return summary


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_evaluation.py:03


def evaluate_recursive_state(
    model: Any,
    tokenizer: Any,
    cfg: dict[str, Any],
    out_dir: Path,
    label: str,
    sampled: bool = True,
) -> dict[str, Any]:
    """Save raw red likelihoods and optional sampled answers per prompt.

    out_dir belongs to this one state. A complete matching evaluation is reused
    only after verifying its protocol, activation, and saved payloads. The
    notebook orchestrator binds saved adapter hashes to this directory. The
    same sampling seed is used for all states, and evaluation restores the
    model's previous training mode and the caller's Torch RNG state.
    """
    import numpy as np
    import torch

    questions = cfg["eval_questions"]
    if (not isinstance(questions, list) or len(questions) != 20
            or any(not isinstance(prompt, str) or not prompt.strip() for prompt in questions)
            or len(set(questions)) != 20):
        raise ValueError("Evaluation requires the same twenty distinct frozen prompts")
    draws, batch_size = cfg["eval_draws"], cfg["eval_batch"]
    if type(draws) is not int or type(batch_size) is not int or draws < 1 or batch_size < 1:
        raise ValueError("Endpoint evaluation requires positive integer draws and batch size")
    if cfg.get("enable_thinking") is not False:
        raise ValueError("This protocol requires enable_thinking=False")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    event = recursive_red_token_event(tokenizer)
    audits = [
        recursive_eval_prompt_audit(
            tokenizer, prompt, render_recursive_prompt(tokenizer, prompt, cfg), event
        ) for prompt in questions
    ]
    # Native EOS and explicit sampling configuration come from the shared cell.
    generation = make_recursive_generation_config(tokenizer, max_new_tokens=32)
    eos_ids = generation.eos_token_id
    eos_ids = [eos_ids] if isinstance(eos_ids, int) else list(eos_ids)
    if not eos_ids or any(type(token) is not int for token in eos_ids):
        raise ValueError("Generation must define the native EOS token IDs")
    if set(eos_ids).intersection(event["token_ids"]):
        raise ValueError("A red content token cannot be a native EOS")
    if (generation.temperature != 1.0 or generation.top_p != 1.0
            or generation.top_k != 0 or generation.repetition_penalty != 1.0
            or generation.do_sample is not True):
        raise ValueError("Evaluation generation configuration differs from the frozen protocol")
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = eos_ids[0]
    activation = recursive_live_adapter_audit(model)
    protocol = {
        "schema": "recursive-sl-red-evaluation-v1",
        "label": label,
        "model_id": cfg["model_id"],
        "model_revision": cfg["model_revision"],
        "config_sha256": digest_object(cfg),
        "red_token_event": event,
        "questions": questions,
        "prompt_audits": audits,
        "neutral_system": None,
        "answer_prefill": None,
        "enable_thinking": False,
        "activation": activation,
        "probability_definition": "Full-vocabulary probability that the first answer token is one of six canonical red tokens; no color-only renormalization",
        "aggregation": "Arithmetic mean of per-prompt probabilities with equal prompt weights",
        "mean_log_definition": "Arithmetic mean of per-prompt natural-log red probabilities",
        "log_mean_definition": "Natural log of the arithmetic mean red probability",
        "sampled": sampled,
        "sample_definition": "Whole stripped reply is red, case insensitive, optionally followed by .!?,;: or ellipsis; must terminate with native EOS",
        "sample_draws_per_prompt": draws if sampled else 0,
        "sampling_seed": int(cfg["eval_seed"]),
        "sampling_batch_size": batch_size,
        "max_new_tokens": 32,
        "generation_config": generation.to_dict(),
        "use_model_defaults": False,
        "probability_cache_dtype": "float32",
    }
    identity = digest_object(protocol)
    completed = verify_recursive_evaluation(out_dir, protocol)
    if completed is not None:
        logger.info("Reusing verified evaluation {}", label)
        return completed
    if (out_dir / "protocol.json").exists():
        previous = json.loads((out_dir / "protocol.json").read_text())
        if digest_object(previous) != identity:
            raise ValueError("Incomplete evaluation directory belongs to a different protocol")
    atomic_json(out_dir / "protocol.json", protocol)
    device = next(model.parameters()).device
    if device.type != "cuda":
        raise ValueError("Notebook model evaluation requires its loaded CUDA model")
    device_index = device.index if device.index is not None else torch.cuda.current_device()
    was_training = model.training
    raw_logits = []
    log_probabilities = []
    samples: list[dict[str, Any]] = []
    counts = [0] * len(questions)
    truncated = [0] * len(questions)

    def encoded_batch(prefixes: list[list[int]]) -> dict[str, Any]:
        """Use explicit left padding without mutating the tokenizer."""
        width = max(map(len, prefixes))
        ids = [[pad_id] * (width - len(prefix)) + prefix for prefix in prefixes]
        masks = [[0] * (width - len(prefix)) + [1] * len(prefix) for prefix in prefixes]
        return {"input_ids": torch.tensor(ids, device=device),
                "attention_mask": torch.tensor(masks, device=device)}

    try:
        model.eval()
        with torch.inference_mode(), torch.random.fork_rng(devices=[device_index]):
            torch.manual_seed(int(cfg["eval_seed"]))
            torch.cuda.manual_seed(int(cfg["eval_seed"]))
            for start in range(0, len(audits), batch_size):
                inputs = encoded_batch([row["prefix_token_ids"]
                                        for row in audits[start:start + batch_size]])
                positions = inputs["attention_mask"].long().cumsum(-1) - 1
                positions.masked_fill_(inputs["attention_mask"] == 0, 0)
                output = model(**inputs, position_ids=positions, use_cache=False, logits_to_keep=1)
                logits = output.logits[:, -1, :].float()
                if not bool(torch.isfinite(logits).all()):
                    raise ValueError("Model returned nonfinite first-answer logits")
                raw_logits.append(logits.cpu().numpy())
                log_probabilities.append(torch.log_softmax(logits, dim=-1).cpu().numpy())
                del output, logits, inputs, positions
            if sampled:
                requests = [(prompt_index, draw_index)
                            for prompt_index in range(len(questions)) for draw_index in range(draws)]
                for start in range(0, len(requests), batch_size):
                    rows = requests[start:start + batch_size]
                    inputs = encoded_batch([audits[index]["prefix_token_ids"] for index, _ in rows])
                    outputs = model.generate(**inputs, generation_config=generation,
                                             use_model_defaults=False)
                    continuation = outputs[:, inputs["input_ids"].shape[1]:].cpu().tolist()
                    for (index, draw), tokens in zip(rows, continuation, strict=True):
                        content, stopped = recursive_trim_at_eos(tokens, eos_ids)
                        text = tokenizer.decode(content, skip_special_tokens=False,
                                                clean_up_tokenization_spaces=False)
                        exact = stopped and is_recursive_exact_red(text)
                        counts[index] += int(exact)
                        truncated[index] += int(not stopped)
                        samples.append({"prompt_index": index, "draw_index": draw,
                                        "prompt": questions[index], "completion": text,
                                        "content_token_ids": content, "generated_token_ids": tokens,
                                        "eos_stopped": stopped, "exact_red": exact})
                    del outputs, continuation, inputs
    finally:
        model.train(was_training)
    full_logits = np.concatenate(raw_logits, axis=0)
    full_log_probs = np.concatenate(log_probabilities, axis=0)
    summary = recursive_probability_summary(full_log_probs, event["token_ids"])
    summary.update({
        "schema": "recursive-sl-red-evaluation-summary-v1",
        "label": label,
        "protocol_sha256": identity,
        "probability_unit": "fraction (multiply by 100 for percent)",
        "sampled_red_rate": sum(counts) / (draws * len(questions)) if sampled else None,
        "sampled_red_counts": counts if sampled else None,
        "sampled_red_total": sum(counts) if sampled else None,
        "sampled_draws_per_prompt": draws if sampled else 0,
        "sampled_total_draws": draws * len(questions) if sampled else 0,
        "sampled_non_eos_counts": truncated if sampled else None,
        "sampled_non_eos_total": sum(truncated) if sampled else None,
        "sampled_red_rate_definition": "Exact EOS-terminated red replies divided by all requested draws, including truncated or noncanonical replies in the denominator",
    })
    temporary_cache = out_dir / "first_token_cache.tmp.npz"
    np.savez_compressed(temporary_cache, first_token_logits=full_logits,
                        first_token_log_probs=full_log_probs,
                        red_token_ids=np.asarray(event["token_ids"], dtype=np.int64))
    cache = out_dir / "first_token_cache.npz"
    temporary_cache.replace(cache)
    atomic_json(out_dir / "summary.json", summary)
    atomic_json(out_dir / "samples.json", samples)
    atomic_json(out_dir / "complete.json", {
        "schema": "recursive-sl-red-evaluation-complete-v1",
        "label": label,
        "protocol_sha256": identity,
        "files": {name: digest_file(out_dir / name) for name in (
            "protocol.json", "first_token_cache.npz", "summary.json", "samples.json")},
    })
    logger.success("{}: mean P(red)={:.8g}; sampled red={}", label,
                   summary["mean_red_probability"], summary["sampled_red_rate"])
    return summary


### recursive_colab_training.py


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_training.py:00
"""Readable Colab helpers for completion-only, resumable recursive SFT.

The notebook defines load_recursive_model, release_recursive_model, atomic_json,
digest_file, digest_object, render_recursive_prompt, and
verify_recursive_evaluation before this cell. No
remote source download or encoded source payload is required. Optimizer state is
reset between experimental legs; an interrupted leg resumes its saved optimizer,
scheduler, data position, and random states. New fast legs can select fused AdamW
and a larger physical batch in their new run configuration. Existing legs keep
their original settings and fingerprint; batching is never changed on resume.
"""


import contextlib
import errno
import gc
import hashlib
import json
import math
from pathlib import Path
import random
import re
import shutil
import tempfile
from typing import Any, Callable, Iterator
import uuid

from loguru import logger


def recursive_numeric_completion(row: dict[str, Any]) -> tuple[str, str]:
    """Validate the user-only prompt and unchanged ten-number response."""
    prompt, completion = row.get("prompt"), row.get("completion")
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Each training row requires a nonempty prompt")
    if not isinstance(completion, str):
        raise ValueError("Each training row requires a completion string")
    if not re.fullmatch(r"\s*[0-9]{2}(?: [0-9]{2}){9}\s*", completion):
        raise ValueError("Completion must contain exactly 10 two-digit numbers")
    if re.search(r"\bred\b|<\|", prompt, flags=re.IGNORECASE):
        raise ValueError("Trait text or chat-control tokens leaked into prompt")
    return prompt, completion


def tokenize_recursive_row(
    tokenizer: Any, row: dict[str, Any], cfg: dict[str, Any]
) -> dict[str, list[int]]:
    """Mask the complete native prefix, supervising only numbers and turn end."""
    prompt, completion = recursive_numeric_completion(row)
    prefix = render_recursive_prompt(tokenizer, prompt, cfg)
    if tokenizer.eos_token_id != 151645:
        raise ValueError("Expected the native Qwen im_end EOS token 151645")
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    answer_ids = tokenizer.encode(prefix + completion, add_special_tokens=False)
    if answer_ids[:len(prefix_ids)] != prefix_ids:
        raise ValueError("Tokenization changed across the prefix/answer boundary")
    input_ids = answer_ids + [tokenizer.eos_token_id]
    if len(input_ids) > int(cfg["max_length"]):
        raise ValueError("A training row exceeds max_length; truncation is forbidden")
    if len(answer_ids) <= len(prefix_ids):
        raise ValueError("Completion produced no supervised number tokens")
    labels = [-100] * len(prefix_ids) + input_ids[len(prefix_ids):]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
            "labels": labels}


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_training.py:01


def recursive_training_plan(
    cfg: dict[str, Any], parent_adapter: Path | None,
    dataset_rows: list[dict[str, Any]], seed: int
) -> dict[str, Any]:
    """Freeze model, parent, dataset, code identity, and leg hyperparameters."""
    if len(dataset_rows) != int(cfg["train_examples"]):
        raise ValueError("Training dataset count differs from the frozen config")
    for row in dataset_rows:
        recursive_numeric_completion(row)
    positive_keys = ("train_examples", "train_epochs", "micro_batch",
                     "gradient_accumulation", "max_length", "checkpoint_every")
    for key in positive_keys:
        if isinstance(cfg[key], bool) or int(cfg[key]) != cfg[key] or cfg[key] < 1:
            raise ValueError(f"{key} must be a positive integer")
    if not re.fullmatch(r"[0-9a-f]{40}", cfg["model_revision"]):
        raise ValueError("Model revision must be an immutable 40-hex commit")
    if not cfg.get("run_fingerprint"):
        raise ValueError("The notebook must supply its frozen run_fingerprint")
    optimizer = cfg.get("optimizer", "adamw_torch")
    if optimizer not in ("adamw_torch", "adamw_torch_fused"):
        raise ValueError("Use native adamw_torch or adamw_torch_fused")
    parent_hashes = None
    if parent_adapter is not None:
        parent_adapter = Path(parent_adapter)
        if parent_adapter.is_symlink() or not parent_adapter.is_dir():
            raise ValueError("Parent adapter must be an existing real directory")
        parent_hashes = {}
        for name in ("adapter_config.json", "adapter_model.safetensors"):
            path = parent_adapter / name
            if path.is_symlink() or not path.is_file():
                raise ValueError(f"Parent adapter is incomplete: {name}")
            parent_hashes[name] = digest_file(path)
    settings = {key: cfg[key] for key in (
        "train_examples", "train_epochs", "micro_batch", "gradient_accumulation",
        "lr", "lora_r", "lora_alpha", "max_length", "checkpoint_every")}
    settings.update({
        "warmup_steps": 5, "weight_decay": 0.0, "max_grad_norm": 1.0,
        "optimizer": optimizer, "lr_scheduler": "linear",
        "group_by_length": True, "packing": False, "truncation": False,
        "loss": "completion plus native im_end; all prompt/empty-thinking tokens masked",
        "precision": "BF16 base and autocast; FP32 trainable adapters",
        "gradient_checkpointing": "nonreentrant", "tf32": False,
        "optimizer_between_legs": "reset", "optimizer_within_interrupted_leg": "resume",
    })
    # Omission is intentional for old plans: adding a new default would change
    # the fingerprint of an interrupted native leg.
    if "training_backend" in cfg:
        if cfg["training_backend"] not in ("native", "unsloth"):
            raise ValueError("training_backend must be native or unsloth")
        settings["training_backend"] = cfg["training_backend"]
    updates = math.ceil(len(dataset_rows) / (
        int(cfg["micro_batch"]) * int(cfg["gradient_accumulation"])))
    plan = {
        "schema": "recursive-completion-sft-v1", "run_fingerprint": cfg["run_fingerprint"],
        "model_id": cfg["model_id"], "model_revision": cfg["model_revision"],
        "dataset_sha256": digest_object(dataset_rows), "parent_sha256": parent_hashes,
        "seed": int(seed), "settings": settings,
        "updates_per_epoch": updates, "total_updates": updates * int(cfg["train_epochs"]),
        "evaluation_rng": "saved and restored around every observation",
    }
    return {**plan, "fingerprint": digest_object(plan)}


@contextlib.contextmanager
def recursive_leg_lock(out_dir: Path, plan: dict[str, Any]) -> Iterator[None]:
    """Lock a leg locally and refuse incompatible or unowned output files."""
    import fcntl

    if out_dir.is_symlink():
        raise ValueError("Leg output cannot be a symlink")
    lock_id = hashlib.sha256(str(out_dir.resolve()).encode()).hexdigest()
    with (Path(tempfile.gettempdir()) / f"recursive-sft-{lock_id}.lock").open("a") as handle:
        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            out_dir.mkdir(parents=True, exist_ok=True)
            if any(path.is_symlink() for path in out_dir.rglob("*")):
                raise ValueError("Leg output contains an unsupported symlink")
            run_path = out_dir / "run.json"
            if run_path.exists():
                if json.loads(run_path.read_text()) != plan:
                    raise ValueError("Leg identity changed; choose a new run directory")
            elif any(out_dir.iterdir()):
                restore_missing_recursive_leg_plan(out_dir, plan)
            else:
                atomic_json(run_path, plan)
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_training.py:02


def recursive_payload_hashes(directory: Path) -> dict[str, str]:
    """Hash every regular checkpoint payload except the integrity seal itself."""
    if directory.is_symlink() or not directory.is_dir():
        raise ValueError("Checkpoint must be a real directory")
    result = {}
    for path in sorted(directory.rglob("*")):
        if path.is_symlink():
            raise ValueError("Checkpoint contains an unsupported symlink")
        if path.is_file() and path.name != "checkpoint_integrity.json":
            result[str(path.relative_to(directory))] = digest_file(path)
    return result


def restore_missing_recursive_leg_plan(out_dir: Path, plan: dict[str, Any]) -> None:
    """Restore only absent metadata from fully verified sealed checkpoints.

    The caller must hold recursive_leg_lock. Recompute the plan from the frozen
    configuration, parent weights, and committed dataset before calling. A
    checkpoint's content-bound fingerprint must prove that exact plan; a folder
    name or an unsealed save is insufficient evidence. No model state is changed.
    """
    run_path = out_dir / "run.json"
    if run_path.is_symlink():
        raise ValueError("Leg plan cannot be a symlink")
    if run_path.exists():
        if json.loads(run_path.read_text()) != plan:
            raise ValueError("Leg identity changed; choose a new run directory")
        return
    if plan.get("fingerprint") != digest_object({
            key: value for key, value in plan.items() if key != "fingerprint"}):
        raise ValueError("Cannot restore a leg plan with an invalid fingerprint")
    required = {"adapter_config.json", "adapter_model.safetensors", "optimizer.pt",
                "scheduler.pt", "trainer_state.json", "rng_state.pth"}
    evidence = []
    for checkpoint in sorted((out_dir / "trainer").glob("checkpoint-*")):
        if (not re.fullmatch(r"checkpoint-[0-9]+", checkpoint.name)
                or checkpoint.is_symlink() or not checkpoint.is_dir()):
            raise ValueError("Cannot restore a plan from an invalid checkpoint path")
        seal_path = checkpoint / "checkpoint_integrity.json"
        if seal_path.is_symlink():
            raise ValueError("Checkpoint integrity seal cannot be a symlink")
        if not seal_path.exists():
            continue  # Interrupted writes remain untouched and supply no proof.
        record = json.loads(seal_path.read_text())
        step = int(checkpoint.name.split("-")[-1])
        payloads = recursive_payload_hashes(checkpoint)
        if (record.get("fingerprint") != plan["fingerprint"]
                or type(record.get("step")) is not int or record["step"] != step
                or record.get("sha256") != payloads or not required.issubset(payloads)
                or not 0 < step <= plan["total_updates"]
                or (step != plan["total_updates"]
                    and step % plan["settings"]["checkpoint_every"])):
            raise ValueError("Missing leg plan has invalid or mismatched checkpoint evidence")
        state = json.loads((checkpoint / "trainer_state.json").read_text())
        if type(state.get("global_step")) is not int or state["global_step"] != step:
            raise ValueError("Missing leg plan checkpoint has a mismatched Trainer step")
        evidence.append({"checkpoint": str(checkpoint.relative_to(out_dir)), "step": step,
                         "checkpoint_integrity_sha256": digest_file(seal_path)})
    if not evidence:
        raise ValueError("Missing run.json cannot be restored without a verified sealed checkpoint")
    serialized = (json.dumps(plan, indent=2, sort_keys=True, allow_nan=False) + "\n").encode()
    expected_hash = hashlib.sha256(serialized).hexdigest()
    completion_path = out_dir / "complete.json"
    if completion_path.exists():
        completion = json.loads(completion_path.read_text())
        if (completion.get("fingerprint") != plan["fingerprint"]
                or completion.get("sha256", {}).get("run.json") != expected_hash):
            raise ValueError("Missing plan bytes disagree with completed-leg proof")
    receipt = {"schema": "recursive-missing-leg-plan-recovery-v1",
               "plan_fingerprint": plan["fingerprint"], "restored_file": "run.json",
               "restored_sha256": expected_hash, "verified_checkpoints": evidence,
               "policy": "Restore only missing run.json after verifying every sealed checkpoint; preserve all model, optimizer, dataset, and evaluation files."}
    receipt_path = out_dir / "execution_repairs" / f"missing-plan-{digest_object(receipt)}.json"
    if receipt_path.exists() and json.loads(receipt_path.read_text()) != receipt:
        raise ValueError("Missing-plan recovery receipt was modified")
    atomic_json(receipt_path, receipt)
    # A concurrent writer or delayed Drive visibility must never replace a
    # newly appeared incompatible plan, even while this process holds its flock.
    if run_path.is_symlink():
        raise ValueError("Leg plan cannot be a symlink")
    if run_path.exists():
        if run_path.read_bytes() != serialized:
            raise ValueError("A different run.json appeared during metadata recovery")
    else:
        atomic_json(run_path, plan)
    if digest_file(run_path) != expected_hash:
        raise ValueError("Restored leg plan failed exact-byte verification")
    logger.warning("Restored only {} from verified checkpoint evidence; receipt: {}",
                   run_path, receipt_path)


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_training.py:03


def digest_recursive_leg_plan(out_dir: Path, plan: dict[str, Any]) -> str:
    """Check final plan identity and retry only an absent plan during sealing.

    Called under the leg lock. Missing data, adapters, evaluations, or optimizer
    payloads never enter this recovery path and still fail their normal checks.
    """
    path = out_dir / "run.json"
    for attempt in range(2):
        try:
            if path.is_symlink() or json.loads(path.read_text()) != plan:
                raise ValueError("Leg identity changed before finalization")
            return digest_file(path)
        except FileNotFoundError as error:
            if (attempt or error.errno != errno.ENOENT or error.filename is None
                    or Path(error.filename).absolute() != path.absolute()):
                raise
            restore_missing_recursive_leg_plan(out_dir, plan)
    raise RuntimeError("Unreachable leg-plan digest state")


def seal_recursive_checkpoint(
    checkpoint: Path, plan: dict[str, Any], step: int
) -> dict[str, Any]:
    """Seal a completed Trainer save, including optimizer/scheduler/RNG state."""
    required = {"adapter_config.json", "adapter_model.safetensors", "optimizer.pt",
                "scheduler.pt", "trainer_state.json", "rng_state.pth"}
    payloads = recursive_payload_hashes(checkpoint)
    if not required.issubset(payloads):
        raise ValueError("Trainer checkpoint lacks complete optimizer/scheduler/RNG state")
    state = json.loads((checkpoint / "trainer_state.json").read_text())
    if state.get("global_step") != step or checkpoint.name != f"checkpoint-{step}":
        raise ValueError("Trainer checkpoint step mismatch")
    record = {"fingerprint": plan["fingerprint"], "step": step, "sha256": payloads}
    path = checkpoint / "checkpoint_integrity.json"
    if path.exists() and json.loads(path.read_text()) != record:
        raise ValueError("A sealed Trainer checkpoint was modified")
    atomic_json(path, record)
    return record


def find_recursive_resume(out_dir: Path, plan: dict[str, Any]) -> Path | None:
    """Resume only sealed checkpoints; preserve interrupted writes separately."""
    candidates = []
    for path in sorted((out_dir / "trainer").glob("checkpoint-*")):
        if not re.fullmatch(r"checkpoint-[0-9]+", path.name) or path.is_symlink():
            raise ValueError("Unexpected Trainer checkpoint path")
        step = int(path.name.split("-")[-1])
        seal = path / "checkpoint_integrity.json"
        if not seal.is_file():
            quarantine = out_dir / "interrupted_checkpoint_writes"
            quarantine.mkdir(exist_ok=True)
            destination = quarantine / f"{path.name}-{uuid.uuid4().hex[:8]}"
            path.rename(destination)
            logger.warning("Preserved an unsealed interrupted save at {}", destination)
            continue
        record = json.loads(seal.read_text())
        if (record.get("fingerprint") != plan["fingerprint"]
                or record.get("step") != step
                or record.get("sha256") != recursive_payload_hashes(path)):
            raise ValueError("Resume checkpoint integrity verification failed")
        if not 0 < step <= plan["total_updates"]:
            raise ValueError("Resume checkpoint lies outside this leg")
        if step != plan["total_updates"] and step % plan["settings"]["checkpoint_every"]:
            raise ValueError("Resume checkpoint is not a planned observation step")
        seal_recursive_checkpoint(path, plan, step)
        candidates.append((step, path))
    return max(candidates)[1] if candidates else None


def publish_recursive_adapter(
    checkpoint: Path, out_dir: Path, plan: dict[str, Any], step: int
) -> Path:
    """Retain immutable middle adapter weights independently of resume rotation."""
    destination = out_dir / "adapters" / f"checkpoint-{step:06d}"
    expected = {name: digest_file(checkpoint / name) for name in
                ("adapter_config.json", "adapter_model.safetensors")}
    metadata = {"fingerprint": plan["fingerprint"], "step": step, "sha256": expected}
    if destination.exists():
        if (json.loads((destination / "adapter_integrity.json").read_text()) != metadata
                or any(digest_file(destination / name) != value
                       for name, value in expected.items())):
            raise ValueError("An immutable middle adapter changed")
        return destination
    destination.parent.mkdir(exist_ok=True)
    temporary = Path(tempfile.mkdtemp(prefix=".publishing-", dir=destination.parent))
    try:
        for name in expected:
            shutil.copyfile(checkpoint / name, temporary / name)
        atomic_json(temporary / "adapter_integrity.json", metadata)
        temporary.rename(destination)
    except BaseException:
        shutil.rmtree(temporary, ignore_errors=True)
        raise
    return destination


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_training.py:04


def verify_recursive_completed(out_dir: Path, plan: dict[str, Any]) -> Path | None:
    """Validate a previously completed leg without loading a GPU model."""
    complete = out_dir / "complete.json"
    if not complete.is_file():
        return None
    record = json.loads(complete.read_text())
    expected_relative = f"adapters/checkpoint-{plan['total_updates']:06d}"
    expected_steps = list(range(plan["settings"]["checkpoint_every"],
                                plan["total_updates"] + 1,
                                plan["settings"]["checkpoint_every"]))
    if plan["total_updates"] not in expected_steps:
        expected_steps.append(plan["total_updates"])
    expected_artifacts = {"run.json", "tokenization_audit.json"}
    for step in expected_steps:
        expected_artifacts.update(f"adapters/checkpoint-{step:06d}/{name}" for name in (
            "adapter_config.json", "adapter_model.safetensors", "adapter_integrity.json"))
        expected_artifacts.update(f"evaluations/checkpoint-{step:06d}/{name}" for name in (
            "_training_evaluation.json", "complete.json", "protocol.json",
            "first_token_cache.npz", "summary.json", "samples.json"))
    if (record.get("fingerprint") != plan["fingerprint"]
            or record.get("final_adapter") != expected_relative
            or record.get("checkpoint_steps") != expected_steps
            or record.get("total_updates") != plan["total_updates"]
            or set(record.get("sha256", {})) != expected_artifacts):
        raise ValueError("Completed leg identity mismatch")
    for name, expected in record["sha256"].items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Invalid completed-leg artifact path")
        path = out_dir / relative
        if not path.is_file() or digest_file(path) != expected:
            raise ValueError(f"Completed-leg artifact changed: {name}")
    for step in expected_steps:
        evaluation_dir = out_dir / "evaluations" / f"checkpoint-{step:06d}"
        saved = json.loads((evaluation_dir / "_training_evaluation.json").read_text())
        verify_recursive_training_observation(evaluation_dir, saved)
    return out_dir / expected_relative


def verify_recursive_training_observation(
    evaluation_dir: Path, saved: dict[str, Any]
) -> dict[str, Any]:
    """Verify the actual probability cache and samples behind a leg observation."""
    result = verify_recursive_evaluation(evaluation_dir)
    if result is None:
        raise ValueError("Training observation lacks a completed evaluation cache")
    if (saved.get("result") != result
            or saved.get("result_sha256") != digest_object(result)
            or saved.get("evaluation_complete_sha256") != digest_file(evaluation_dir / "complete.json")):
        raise ValueError("Training observation disagrees with its actual evaluation payloads")
    return result


In [ ]:
# RECURSIVE_VISIBLE_SOURCE: recursive_colab_training.py:05


def train_recursive_leg(
    cfg: dict[str, Any], parent_adapter: Path | None,
    dataset_rows: list[dict[str, Any]], out_dir: Path, seed: int,
    evaluate_callback: Callable[[Any, Any, Path, str, bool], dict[str, Any]],
) -> Path:
    """Train one leg, retaining observations and resuming interrupted SFT safely.

    evaluate_callback receives (model, tokenizer, output_dir, label, sampled).
    It evaluates probabilities at every checkpoint; sampled=True is passed at
    the final checkpoint. The callback must not change adapter parameters.
    """
    out_dir = Path(out_dir)
    plan = recursive_training_plan(cfg, parent_adapter, dataset_rows, seed)
    with recursive_leg_lock(out_dir, plan):
        completed = verify_recursive_completed(out_dir, plan)
        if completed is not None:
            logger.info("Reusing verified completed training leg {}", out_dir)
            return completed
        resume = find_recursive_resume(out_dir, plan)
        resume_step = 0 if resume is None else int(resume.name.split("-")[-1])
        # GPU libraries are intentionally imported only after the CPU guards.
        import numpy as np
        import torch
        from transformers import Trainer, TrainerCallback, TrainingArguments, set_seed
        from transformers.trainer_callback import PrinterCallback, ProgressCallback

        set_seed(seed)
        model = tokenizer = trainer = callback = None
        try:
            model, tokenizer = load_recursive_model(
                cfg, resume if resume is not None else parent_adapter, trainable=True)
            trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
            if not trainable or any(parameter.dtype != torch.float32 for parameter in trainable):
                raise ValueError("Expected nonempty FP32 trainable LoRA parameters")
            del trainable
            model.config.use_cache = False
            tokenized = [tokenize_recursive_row(tokenizer, row, cfg) for row in dataset_rows]
            tokenization_audit = {
                "rows": len(tokenized), "prefix_fully_masked": True, "eos_id": 151645,
                "max_tokens": max(len(row["input_ids"]) for row in tokenized),
                "total_supervised_tokens": sum(sum(v != -100 for v in row["labels"])
                                               for row in tokenized),
                "first_row": tokenized[0], "tokenized_sha256": digest_object(tokenized),
            }
            audit_path = out_dir / "tokenization_audit.json"
            if audit_path.exists() and json.loads(audit_path.read_text()) != tokenization_audit:
                raise ValueError("Tokenization changed during an interrupted training leg")
            atomic_json(audit_path, tokenization_audit)

            class NumericDataset:
                """Unpacked pretokenized completion-only training examples."""

                def __len__(self) -> int:
                    return len(tokenized)

                def __getitem__(self, index: int) -> dict[str, list[int]]:
                    return tokenized[index]

            def collate(rows: list[dict[str, list[int]]]) -> dict[str, Any]:
                """Right-pad inputs while masking padding and prompt labels."""
                maximum = max(len(row["input_ids"]) for row in rows)
                pad_id = tokenizer.pad_token_id
                if pad_id is None:
                    pad_id = tokenizer.eos_token_id
                return {name: torch.tensor([
                    row[name] + [fill] * (maximum - len(row[name])) for row in rows
                ], dtype=torch.long) for name, fill in
                    (("input_ids", pad_id), ("attention_mask", 0), ("labels", -100))}

            def observe(step: int, checkpoint: Path) -> Path:
                """Publish and evaluate an observation without perturbing training RNG."""
                adapter = publish_recursive_adapter(checkpoint, out_dir, plan, step)
                label = f"checkpoint-{step:06d}"
                evaluation_dir = out_dir / "evaluations" / label
                marker = evaluation_dir / "_training_evaluation.json"
                identity = {"fingerprint": plan["fingerprint"], "step": step,
                            "adapter_sha256": digest_file(adapter / "adapter_model.safetensors"),
                            "sampled": step == plan["total_updates"]}
                if marker.exists():
                    saved = json.loads(marker.read_text())
                    if saved.get("identity") != identity or saved.get("result_sha256") != digest_object(saved.get("result")):
                        raise ValueError("Saved checkpoint evaluation identity/integrity mismatch")
                    verify_recursive_training_observation(evaluation_dir, saved)
                    return adapter
                if evaluation_dir.exists() and any(evaluation_dir.iterdir()):
                    partial = out_dir / "interrupted_evaluations"
                    partial.mkdir(exist_ok=True)
                    evaluation_dir.rename(partial / f"{label}-{uuid.uuid4().hex[:8]}")
                evaluation_dir.mkdir(parents=True, exist_ok=True)
                rng = (random.getstate(), np.random.get_state(), torch.get_rng_state(),
                       torch.cuda.get_rng_state_all())
                was_training, use_cache = model.training, model.config.use_cache
                try:
                    result = evaluate_callback(model, tokenizer, evaluation_dir, label,
                                               step == plan["total_updates"])
                    verified = verify_recursive_evaluation(evaluation_dir)
                    if verified is None or verified != result:
                        raise ValueError("Evaluation callback did not save matching, verified payloads")
                    atomic_json(marker, {
                        "identity": identity, "result": result,
                        "result_sha256": digest_object(result),
                        "evaluation_complete_sha256": digest_file(evaluation_dir / "complete.json"),
                    })
                finally:
                    random.setstate(rng[0])
                    np.random.set_state(rng[1])
                    torch.set_rng_state(rng[2])
                    torch.cuda.set_rng_state_all(rng[3])
                    model.train(was_training)
                    model.config.use_cache = use_cache
                    gc.collect()
                    torch.cuda.empty_cache()
                return adapter

            if resume is not None:
                logger.info("Resuming leg {} with optimizer/RNG state at step {}", out_dir.name, resume_step)
                observe(resume_step, resume)
            else:
                logger.info("Starting leg {}; optimizer reset, parent weights preserved", out_dir.name)
            logger.info(
                "{}: physical batch {} × accumulation {}; {}; {} planned updates",
                out_dir.name, cfg["micro_batch"], cfg["gradient_accumulation"],
                plan["settings"]["optimizer"], plan["total_updates"],
            )

            class RecursiveSaveCallback(TrainerCallback):
                """Save the final update and evaluate only after full checkpoint writes."""

                def on_step_end(self, args: Any, state: Any, control: Any, **kwargs: Any) -> Any:
                    if state.global_step == plan["total_updates"]:
                        control.should_save = True
                    return control

                def on_save(self, args: Any, state: Any, control: Any, **kwargs: Any) -> Any:
                    checkpoint = out_dir / "trainer" / f"checkpoint-{state.global_step}"
                    seal_recursive_checkpoint(checkpoint, plan, state.global_step)
                    observe(state.global_step, checkpoint)
                    return control

                def on_log(self, args: Any, state: Any, control: Any,
                           logs: dict[str, Any] | None = None, **kwargs: Any) -> Any:
                    logger.info("{} step {}: {}", out_dir.name, state.global_step, logs or {})
                    return control

            if resume_step < plan["total_updates"]:
                args = TrainingArguments(
                    output_dir=str(out_dir / "trainer"),
                    num_train_epochs=cfg["train_epochs"], max_steps=plan["total_updates"],
                    per_device_train_batch_size=cfg["micro_batch"],
                    gradient_accumulation_steps=cfg["gradient_accumulation"],
                    learning_rate=cfg["lr"], lr_scheduler_type="linear", warmup_steps=5,
                    optim=plan["settings"]["optimizer"], weight_decay=0.0,
                    max_grad_norm=1.0,
                    seed=seed, data_seed=seed, group_by_length=True,
                    dataloader_drop_last=False, dataloader_num_workers=0,
                    bf16=True, fp16=False, tf32=False,
                    gradient_checkpointing=True,
                    gradient_checkpointing_kwargs={"use_reentrant": False},
                    save_strategy="steps", save_steps=cfg["checkpoint_every"],
                    save_total_limit=2, save_only_model=False, save_safetensors=True,
                    eval_strategy="no", logging_strategy="steps", logging_steps=10,
                    logging_first_step=True, report_to=[], disable_tqdm=True,
                    remove_unused_columns=False, label_names=["labels"],
                    ignore_data_skip=False,
                )
                callback = RecursiveSaveCallback()
                trainer = Trainer(model=model, args=args, train_dataset=NumericDataset(),
                                  data_collator=collate, callbacks=[callback])
                trainer.remove_callback(PrinterCallback)
                trainer.remove_callback(ProgressCallback)
                trainer.train(resume_from_checkpoint=str(resume) if resume is not None else None)
                if trainer.state.global_step != plan["total_updates"]:
                    raise ValueError("Trainer did not reach the planned final update")
            final_checkpoint = out_dir / "trainer" / f"checkpoint-{plan['total_updates']}"
            seal_recursive_checkpoint(final_checkpoint, plan, plan["total_updates"])
            final_adapter = observe(plan["total_updates"], final_checkpoint)
            expected_steps = list(range(cfg["checkpoint_every"], plan["total_updates"] + 1,
                                        cfg["checkpoint_every"]))
            if plan["total_updates"] not in expected_steps:
                expected_steps.append(plan["total_updates"])
            protected = [out_dir / "run.json", audit_path]
            for step in expected_steps:
                directory = out_dir / "adapters" / f"checkpoint-{step:06d}"
                protected.extend(directory / name for name in (
                    "adapter_config.json", "adapter_model.safetensors", "adapter_integrity.json"))
                evaluation_dir = out_dir / "evaluations" / f"checkpoint-{step:06d}"
                protected.extend(evaluation_dir / name for name in (
                    "_training_evaluation.json", "complete.json", "protocol.json",
                    "first_token_cache.npz", "summary.json", "samples.json"))
            checksums = {
                str(path.relative_to(out_dir)): (
                    digest_recursive_leg_plan(out_dir, plan)
                    if path == out_dir / "run.json" else digest_file(path))
                for path in protected
            }
            atomic_json(out_dir / "complete.json", {
                "fingerprint": plan["fingerprint"], "final_adapter": str(final_adapter.relative_to(out_dir)),
                "total_updates": plan["total_updates"], "checkpoint_steps": expected_steps,
                "sha256": checksums,
            })
            logger.success("Completed recursive leg {} at step {}", out_dir.name, plan["total_updates"])
            return final_adapter
        finally:
            trainer = callback = None
            if model is not None:
                release_recursive_model(model)
            model = tokenizer = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


### Isolated generation worker


In [ ]:
# RECURSIVE_VLLM_SOURCE: recursive_colab_vllm.py:00
"""Generate one recursive numeric dataset in an isolated vLLM process.

The notebook owns the generation plan and final dataset publication. This worker
verifies its committed prefix, writes compatible atomic raw batches, and exits
after enough valid candidates exist. It never loads training libraries, repairs
responses, retries a rejected prompt, or changes the sampling vocabulary.
"""


import argparse
import hashlib
import json
from pathlib import Path
import re
from typing import Any, Callable

from loguru import logger


vllm_worker_vllm_version = "0.10.0"
vllm_worker_max_num_seqs = 128
vllm_worker_max_num_batched_tokens = 4096
vllm_worker_eos_ids = (151645, 151643)
vllm_worker_number_response = re.compile(r"[ \t\r\n]*[0-9]{2}(?: [0-9]{2}){9}[ \t\r\n]*")
vllm_worker_models = {"Qwen/Qwen2.5-7B-Instruct": "qwen2", "Qwen/Qwen3-8B": "qwen3"}


def vllm_worker_digest_object(value: Any) -> str:
    """Match the notebook's canonical finite JSON fingerprint."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def vllm_worker_digest_file(path: Path) -> str:
    """Hash one local payload without allocating the whole file."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def vllm_worker_read_json(path: Path) -> Any:
    """Read one JSON protocol or committed batch."""
    return json.loads(path.read_text())


def vllm_worker_atomic_json(path: Path, value: Any) -> None:
    """Publish a complete record, never a partly written batch."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".writing")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    temporary.replace(path)


def vllm_worker_recursive_seed(seed: int, *parts: Any) -> int:
    """Derive the same deterministic streams as the notebook."""
    return int(vllm_worker_digest_object([seed, *parts])[:8], 16) % (2**31 - 1)


def vllm_worker_candidate_seed(seed: int, index: int) -> int:
    """Give each candidate an explicit seed independent of scheduler timing."""
    return vllm_worker_recursive_seed(seed, "candidate", index)


def vllm_worker_validate_system_prompt(system_prompt: str | None) -> None:
    """Validate a teacher-only instruction before loading or using an engine."""
    if system_prompt is not None and (
        not isinstance(system_prompt, str) or not system_prompt.strip()
        or re.search(r"<\|", system_prompt)
    ):
        raise ValueError("Generation system_prompt must be nonempty text without chat-control tokens")


def vllm_worker_render_prompt_ids(
    tokenizer: Any, prompt: str, cfg: dict[str, Any], *, system_prompt: str | None = None,
) -> list[int]:
    """Render the native teacher prefix with an optional generation-only trait."""
    vllm_worker_validate_system_prompt(system_prompt)
    kwargs = {"enable_thinking": False} if cfg["model_id"] == "Qwen/Qwen3-8B" else {}
    messages = [{"role": "user", "content": prompt}]
    if system_prompt is not None:
        messages.insert(0, {"role": "system", "content": system_prompt})
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, **kwargs,
    )
    ids = tokenizer.encode(rendered, add_special_tokens=False)
    if not ids or len(ids) + 64 > int(cfg.get("max_length", 512)):
        raise ValueError("Candidate prompt cannot fit unchanged with its 64-token budget")
    return ids


In [ ]:
# RECURSIVE_VLLM_SOURCE: recursive_colab_vllm.py:01


def vllm_worker_normalize_output(output: Any, prompt_index: int, sample_seed: int) -> dict[str, Any]:
    """Retain vLLM's exact response and raw IDs, including invalid responses."""
    if len(output.outputs) != 1:
        raise ValueError("vLLM must return exactly one completion per candidate")
    sample = output.outputs[0]
    text, ids = sample.text, list(sample.token_ids)
    finish, stop = sample.finish_reason, sample.stop_reason
    if (not isinstance(text, str) or len(ids) > 64
            or any(type(token) is not int or token < 0 for token in ids)):
        raise ValueError("vLLM returned malformed raw completion evidence")
    # EOS is sometimes omitted from output.token_ids. Record the finish reason
    # separately; do not invent or append a token to make it look present.
    terminated = finish == "stop" and (stop is None or type(stop) is int and stop in vllm_worker_eos_ids)
    accepted = terminated and vllm_worker_number_response.fullmatch(text) is not None
    return {
        "prompt_index": prompt_index, "text": text,
        "generated_token_ids": ids, "finish_reason": finish, "stop_reason": stop,
        "terminated": terminated, "accepted": accepted,
        "reason": "accepted" if accepted else "format" if terminated else "max_tokens",
        "vllm_worker_candidate_seed": sample_seed,
    }


def vllm_worker_validate_attempt(attempt: dict[str, Any], index: int, seed: int) -> None:
    """Recompute a saved candidate's format decision without any model loading."""
    ids, text = attempt.get("generated_token_ids"), attempt.get("text")
    stop, finish = attempt.get("stop_reason"), attempt.get("finish_reason")
    if (attempt.get("prompt_index") != index or not isinstance(text, str)
            or not isinstance(ids, list) or len(ids) > 64
            or any(type(token) is not int or token < 0 for token in ids)
            or finish not in ("stop", "length")
            or not (stop is None or type(stop) is int and stop in vllm_worker_eos_ids)
            or attempt.get("vllm_worker_candidate_seed") != vllm_worker_candidate_seed(seed, index)):
        raise ValueError("Cached vLLM candidate evidence changed")
    terminated = finish == "stop"
    accepted = terminated and vllm_worker_number_response.fullmatch(text) is not None
    reason = "accepted" if accepted else "format" if terminated else "max_tokens"
    if (attempt.get("terminated") is not terminated
            or attempt.get("accepted") is not accepted or attempt.get("reason") != reason):
        raise ValueError("Cached vLLM candidate filtering changed")


In [ ]:
# RECURSIVE_VLLM_SOURCE: recursive_colab_vllm.py:02


def vllm_worker_inspect_prefix(request: dict[str, Any]) -> tuple[int, int]:
    """Require the exact owned generation plan and a contiguous sealed prefix."""
    cfg = request["cfg"]
    prompts = request["prompts"]
    system_prompt = request.get("system_prompt")
    vllm_worker_validate_system_prompt(system_prompt)
    target, batch = int(cfg["train_examples"]), int(cfg["generation_batch"])
    if (cfg["model_id"] not in vllm_worker_models
            or re.fullmatch(r"[0-9a-f]{40}", cfg["model_revision"]) is None
            or not 0 < target <= len(prompts) == int(cfg["generation_max_requests"])
            or not 1 <= batch <= 512 or len(set(prompts)) != len(prompts)
            or any(not isinstance(prompt, str) or not prompt.strip()
                   or re.search(r"\bred\b|<\|im_", prompt, re.IGNORECASE) for prompt in prompts)):
        raise ValueError("Invalid frozen vLLM generation request")
    directory = Path(request["out_dir"])
    if directory.is_symlink():
        raise ValueError("Generation directory cannot be a symlink")
    plan = vllm_worker_read_json(directory / "plan.json")
    if (vllm_worker_digest_object(plan) != request["plan_hash"] or plan.get("run") != cfg["run_fingerprint"]
            or plan.get("prompts") != vllm_worker_digest_object(prompts) or plan.get("seed") != request["seed"]
            or plan.get("count") != target or plan.get("batch_size") != batch
            or plan.get("system_prompt") != system_prompt):
        raise ValueError("Worker request differs from the owned generation plan")
    adapter = request.get("adapter_path")
    teacher = {"kind": "neutral_base"}
    if adapter is not None:
        adapter_path = Path(adapter)
        if adapter_path.is_symlink() or not adapter_path.is_dir():
            raise ValueError("Teacher adapter must be an existing local directory")
        names = ("adapter_config.json", "adapter_model.safetensors")
        if any((adapter_path / name).is_symlink() for name in names):
            raise ValueError("Teacher adapter payload cannot be a symlink")
        teacher = {"kind": "adapter", "files": {
            name: vllm_worker_digest_file(adapter_path / name) for name in names}}
        adapter_cfg = vllm_worker_read_json(adapter_path / "adapter_config.json")
        if (adapter_cfg.get("r") != 8 or adapter_cfg.get("lora_alpha") != 32
                or adapter_cfg.get("base_model_name_or_path") != cfg["model_id"]
                or adapter_cfg.get("revision") != cfg["model_revision"]):
            raise ValueError("Teacher adapter configuration differs from the pinned model")
    if plan.get("teacher") != teacher:
        raise ValueError("Teacher adapter differs from the owned generation plan")
    attempts, valid = 0, 0
    for position, path in enumerate(sorted((directory / "batches").glob("*.json"))):
        start = position * batch
        state = vllm_worker_read_json(path)
        payload = {key: value for key, value in state.items() if key != "payload_sha256"}
        stop = min(start + batch, len(prompts))
        if (path.is_symlink() or path.name != f"{start:06d}.json" or start >= len(prompts)
                or valid >= target or state.get("payload_sha256") != vllm_worker_digest_object(payload)
                or state.get("plan") != request["plan_hash"] or state.get("start") != start
                or state.get("stop") != stop
                or state.get("sample_seed") != vllm_worker_recursive_seed(request["seed"], "batch", start)
                or len(state.get("attempts", [])) != stop - start):
            raise ValueError("Cached vLLM batches are not the sealed stopping prefix")
        for index, attempt in enumerate(state["attempts"], start=start):
            vllm_worker_validate_attempt(attempt, index, request["seed"])
            valid += int(attempt["accepted"])
        attempts += len(state["attempts"])
    if attempts != request["start_index"] or valid != request["valid_count"]:
        raise ValueError("Worker resume counts differ from committed raw evidence")
    return attempts, valid


In [ ]:
# RECURSIVE_VLLM_SOURCE: recursive_colab_vllm.py:03


class VllmWorkerRuntime:
    """Keep exactly one pinned base and optional local teacher adapter resident."""

    def __init__(self, cfg: dict[str, Any], adapter: str | None, seed: int) -> None:
        import vllm
        from transformers import AutoConfig, AutoTokenizer
        from vllm.lora.request import LoRARequest

        if vllm.__version__ != vllm_worker_vllm_version:
            raise ValueError(f"Expected vLLM {vllm_worker_vllm_version}; received {vllm.__version__}")
        revision = cfg["model_revision"]
        model_cfg = AutoConfig.from_pretrained(cfg["model_id"], revision=revision)
        if model_cfg.model_type != vllm_worker_models[cfg["model_id"]]:
            raise ValueError("Pinned generation model has the wrong architecture")
        self.tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], revision=revision)
        if self.tokenizer.eos_token_id != 151645:
            raise ValueError("Expected Qwen's native end-of-turn token")
        self.engine = vllm.LLM(
            model=cfg["model_id"], revision=revision,
            tokenizer=cfg["model_id"], tokenizer_revision=revision,
            dtype="bfloat16", max_model_len=int(cfg.get("max_length", 512)),
            gpu_memory_utilization=0.85, enable_prefix_caching=True,
            max_num_seqs=min(vllm_worker_max_num_seqs, int(cfg["generation_batch"])),
            max_num_batched_tokens=vllm_worker_max_num_batched_tokens, seed=seed,
            generation_config="vllm", enable_lora=True,
            max_lora_rank=8, max_loras=1, max_cpu_loras=1,
            trust_remote_code=False,
        )
        self.lora = LoRARequest("recursive_teacher", 1, adapter) if adapter is not None else None
        self.evidence = {
            "backend": "vllm", "version": vllm.__version__,
            "model_id": cfg["model_id"], "model_revision": revision,
            "dtype": "bfloat16", "adapter_mode": "local_unmerged_lora",
            "engine_policy": "vLLM 0.10.0 default engine; offline synchronous generate",
            "engine_class": type(getattr(self.engine, "llm_engine", self.engine)).__module__,
            "max_num_seqs": min(vllm_worker_max_num_seqs, int(cfg["generation_batch"])),
            "max_num_batched_tokens": vllm_worker_max_num_batched_tokens,
            "candidate_batch_size": int(cfg["generation_batch"]),
            "gpu_memory_utilization": 0.85, "engine_seed": seed,
            "max_model_len": int(cfg.get("max_length", 512)),
            "seed_policy": "vllm_worker_recursive_seed(leg_seed, 'candidate', candidate_index)",
            "chat_template_sha256": vllm_worker_digest_object(self.tokenizer.chat_template),
            "sampling": {"temperature": 1.0, "top_p": 1.0, "top_k": -1,
                         "repetition_penalty": 1.0, "max_tokens": 64,
                         "stop_token_ids": list(vllm_worker_eos_ids), "stop": None,
                         "generation_config": "vllm", "guided_decoding": None,
                         "skip_special_tokens": False, "include_stop_str_in_output": False},
        }

    def generate(self, prompt_ids: list[list[int]], seeds: list[int]) -> list[Any]:
        """Sample exactly once per prompt using full-distribution decoding."""
        from vllm import SamplingParams

        params = [SamplingParams(
            n=1, temperature=1.0, top_p=1.0, top_k=-1,
            repetition_penalty=1.0, presence_penalty=0.0, frequency_penalty=0.0,
            max_tokens=64, min_tokens=0, seed=seed,
            stop_token_ids=list(vllm_worker_eos_ids), stop=None, ignore_eos=False,
            skip_special_tokens=False, spaces_between_special_tokens=False,
            include_stop_str_in_output=False,
        ) for seed in seeds]
        return self.engine.generate(
            [{"prompt_token_ids": ids} for ids in prompt_ids],
            sampling_params=params, lora_request=self.lora, use_tqdm=False,
        )

    def close(self) -> None:
        """Shut down the pinned engine's worker before the CLI process exits."""
        engine = getattr(self.engine, "llm_engine", None)
        core = getattr(engine, "engine_core", None)
        executor = getattr(engine, "model_executor", None)
        shutdown = getattr(core, "shutdown", None) or getattr(executor, "shutdown", None)
        if callable(shutdown):
            shutdown()
        self.engine = None


In [ ]:
# RECURSIVE_VLLM_SOURCE: recursive_colab_vllm.py:04


def vllm_worker_run_generation(
    request: dict[str, Any], *, runtime_factory: Callable[..., Any] = VllmWorkerRuntime,
) -> dict[str, Any]:
    """Generate all missing candidate batches with one engine, then exit."""
    start, valid = vllm_worker_inspect_prefix(request)
    cfg, prompts = request["cfg"], request["prompts"]
    target, batch = int(cfg["train_examples"]), int(cfg["generation_batch"])
    directory = Path(request["out_dir"])
    runtime_path = directory / "vllm_runtime.json"
    prior_runtime = vllm_worker_read_json(runtime_path) if runtime_path.is_file() else None
    if start and (not isinstance(prior_runtime, dict)
                  or prior_runtime.get("backend") != "vllm"
                  or prior_runtime.get("version") != vllm_worker_vllm_version):
        raise ValueError("Committed vLLM batches are missing their runtime evidence")
    if valid >= target:
        return {"plan": request["plan_hash"], "attempts": start, "valid_candidates": valid,
                "target": target, "complete": True, "reused_complete_prefix": True,
                "runtime": prior_runtime}
    if start == len(prompts):
        raise RuntimeError(f"Candidate pool exhausted: {valid}/{target} valid responses from {start} candidates")
    runtime = runtime_factory(cfg, request.get("adapter_path"), request["seed"])
    try:
        if runtime_path.exists() and vllm_worker_read_json(runtime_path) != runtime.evidence:
            raise ValueError("vLLM runtime evidence differs from this stage's previous generation")
        vllm_worker_atomic_json(runtime_path, runtime.evidence)
        for index in range(start, len(prompts), batch):
            stop = min(index + batch, len(prompts))
            ids = [vllm_worker_render_prompt_ids(
                runtime.tokenizer, prompt, cfg, system_prompt=request.get("system_prompt"),
            ) for prompt in prompts[index:stop]]
            seeds = [vllm_worker_candidate_seed(request["seed"], i) for i in range(index, stop)]
            outputs = runtime.generate(ids, seeds)
            if len(outputs) != len(ids):
                raise ValueError("vLLM did not return exactly one output for every candidate")
            # vLLM's offline API returns input order. Check the echoed token IDs so
            # candidate/response alignment is proven before writing a batch.
            if any(list(output.prompt_token_ids) != tokens
                   for output, tokens in zip(outputs, ids, strict=True)):
                raise ValueError("vLLM returned responses in a different prompt order")
            attempts = [vllm_worker_normalize_output(output, i, sample_seed)
                        for i, output, sample_seed in zip(range(index, stop), outputs, seeds, strict=True)]
            for i, attempt in enumerate(attempts, start=index):
                vllm_worker_validate_attempt(attempt, i, request["seed"])
            state = {"plan": request["plan_hash"], "start": index, "stop": stop,
                     "sample_seed": vllm_worker_recursive_seed(request["seed"], "batch", index),
                     "attempts": attempts}
            state["payload_sha256"] = vllm_worker_digest_object(state)
            destination = directory / "batches" / f"{index:06d}.json"
            if destination.exists():
                raise ValueError("Refusing to overwrite an existing raw candidate batch")
            vllm_worker_atomic_json(destination, state)
            valid += sum(int(attempt["accepted"]) for attempt in attempts)
            start = stop
            logger.info("{}: {} valid from {} candidates; target {}", directory.name, valid, start, target)
            if valid >= target:
                break
        summary = {"plan": request["plan_hash"], "attempts": start, "valid_candidates": valid,
                   "target": target, "complete": valid >= target, "runtime": runtime.evidence}
        if valid < target:
            raise RuntimeError(f"Candidate pool exhausted: {valid}/{target} valid responses from {start} candidates")
        return summary
    finally:
        close = getattr(runtime, "close", None)
        if callable(close):
            close()


def vllm_worker_main() -> None:
    """Run the isolated worker and publish a compact success or failure receipt."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--request-json", required=True, type=Path)
    parser.add_argument("--output-json", required=True, type=Path)
    args = parser.parse_args()
    try:
        result = vllm_worker_run_generation(vllm_worker_read_json(args.request_json))
    except Exception as error:
        vllm_worker_atomic_json(args.output_json, {"complete": False, "error_type": type(error).__name__, "error": str(error)})
        logger.exception("vLLM generation failed; committed raw batches remain reusable")
        raise
    vllm_worker_atomic_json(args.output_json, result)
    logger.success("Generation complete: {} valid candidates from {} requests", result["valid_candidates"], result["attempts"])


In [ ]:
worker_history = get_ipython().history_manager.input_hist_raw
worker_cells = {source.splitlines()[0]: source for source in worker_history
                if source.startswith("# RECURSIVE_VLLM_SOURCE:")}
if not worker_cells:
    raise RuntimeError("Run the visible generation-worker cells first")
worker_source = "from __future__ import annotations\n" + "\n".join(
    worker_cells[key] for key in sorted(worker_cells))
worker_source += "\nif __name__ == '__main__':\n    vllm_worker_main()\n"
worker_path = Path(CFG["vllm_worker_path"])
compile(worker_source, str(worker_path), "exec")
worker_path.write_text(worker_source)
logger.info("Saved executed worker source to {}", worker_path)


## 4. Freeze configuration, source and runtime; verify any saved run


In [ ]:
CFG, A0 = prepare_recursive_run(CFG, PROJECT, RUN)
logger.success("Ready: {}; base initialization: {}", CFG["model_id"], A0 is None)


## 5. Run or resume
Long-running cell; saved work is checked before reuse.


In [ ]:
sink = logger.add(str(RUN / "experiment.log"), enqueue=False)
try:
    run_recursive_experiment(CFG, A0, RUN)
except Exception:
    logger.exception("Experiment stopped; preserve saved files for resume")
    raise
finally:
    logger.remove(sink)


## 6. Inspect saved results
Report only; no model generation or training.


In [ ]:
build_recursive_report(RUN)
import csv
from IPython.display import Image, Markdown, display

with (RUN / "report/all_checkpoints.csv").open() as handle:
    observations = list(csv.DictReader(handle))
comparisons = []
for seed in CFG["seeds"]:
    endpoints = {row["state"]: row for row in observations
                 if int(row["seed"]) == seed and row["sampled"] == "True"}
    pairs = [("B0 (unmodified base)", "base", "base"),
             ("B1 (shared)", "B1_shared", "B1_shared")]
    pairs += [(f"B{r}", f"B{r}_reciprocal", f"B{r}_frozen_A")
              for r in range(2, CFG["feedback_rounds"] + 2)]
    table = ["| State | Reciprocal P(red) | Frozen-A P(red) | Difference (pp) |",
             "|---|---:|---:|---:|"]
    for label, recursive_name, frozen_name in pairs:
        rec, frozen = endpoints.get(recursive_name), endpoints.get(frozen_name)
        if rec is None or frozen is None:
            table.append(f"| {label} | pending | pending | pending |")
            continue
        p, q = float(rec["mean_red_probability"]), float(frozen["mean_red_probability"])
        comparisons.append({"seed": seed, "state": label, "reciprocal_probability": p,
                            "frozen_A_probability": q, "difference": p - q})
        table.append(f"| {label} | {100*p:.6g}% | {100*q:.6g}% | {100*(p-q):+.6g} |")
    display(Markdown(f"**Seed {seed}: neutral evaluation**\n\n" + "\n".join(table)))
if comparisons:
    with (RUN / "report/B_comparison.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparisons[0]))
        writer.writeheader()
        writer.writerows(comparisons)
display(Image(filename=str(RUN / "report/B_trajectory.png")))
logger.info("All intermediate observations: {}", RUN / "report/all_checkpoints.csv")


## 7. Flush Drive and release the GPU
Runs only after successful completion.


In [ ]:
from google.colab import drive, runtime
for directory in [RUN]:
    if not (directory / "complete.json").is_file():
        raise RuntimeError(f"Experiment incomplete: {directory}")
logger.success("Selected runs complete; flushing Drive")
drive.flush_and_unmount()
runtime.unassign()
